# LeetCode 301 — Remove Invalid Parentheses (Hard) bằng Python

Notebook này giảng **từ căn bản** một bài "dấu ngoặc" kinh điển mức Hard. Skeleton trên LeetCode (Python3):

```python
class Solution:
    def removeInvalidParentheses(self, s: str) -> List[str]:

```

---

## Sau notebook này, bạn sẽ nắm được

1. Kiểm tra chuỗi ngoặc hợp lệ bằng một **biến đếm `balance`** (ôn lại #20 Valid Parentheses, #1249) — kể cả khi chuỗi có **chữ cái**.
2. **Brute force bằng bitmask**: thử mọi cách giữ/xóa ký tự. Chậm, nhưng chắc chắn đúng → dùng làm **"trọng tài"** để kiểm tra các cách sau.
3. **BFS theo tầng** (Breadth-First Search — tìm kiếm theo chiều rộng): xóa 0 ký tự, rồi 1, rồi 2... và *vì sao* tầng đầu tiên có chuỗi hợp lệ chính là đáp án "xóa ít nhất"; vai trò của tập `visited`.
4. **Đếm trước** số `'('` và `')'` *bắt buộc* phải xóa (`left_rem`, `right_rem`) bằng **một lượt quét**, rồi **DFS / backtracking** xóa đúng chừng đó, có **cắt tỉa** (pruning) và **bỏ trùng** — đây là **cách chính**.
5. **DFS hai chiều** với hai biến nhớ `last_i`, `last_j` và mẹo **đảo chuỗi** — lời giải sáng tạo nổi tiếng, không cần `set` mà vẫn không sinh trùng.
6. Kiểm thử chéo với brute force trên chuỗi ngẫu nhiên có chữ cái, lỗi hay gặp, mẹo nhận dạng pattern và **6 bài tập** có đáp án.

Lộ trình: `Bitmask O(2^n · n)` → `BFS theo tầng` → `Đếm + DFS cắt tỉa` → `DFS hai chiều`.

Ký hiệu dùng xuyên suốt notebook: **`n = len(s)`** (tối đa 25) và **`p` = số ký tự ngoặc** trong `s` (tối đa 20).

## 1. Đề bài

Cho một chuỗi `s` gồm các dấu ngoặc `'('`, `')'` và chữ cái. Hãy **xóa ít nhất** số dấu ngoặc không hợp lệ để chuỗi trở thành **hợp lệ**.

Trả về **danh sách tất cả các chuỗi hợp lệ khác nhau** (unique) đạt được với **số lần xóa ít nhất** đó. Thứ tự các chuỗi trong danh sách **tùy ý**.

> Gốc tiếng Anh: *Given a string `s` that contains parentheses and letters, remove the minimum number of invalid parentheses to make the input string valid. Return a list of **unique strings** that are valid with the minimum number of removals. You may return the answer in **any order**.*

### Ví dụ trong đề

| | Input `s` | Output |
|---|---|---|
| Ví dụ 1 | `"()())()"` | `["(())()","()()()"]` |
| Ví dụ 2 | `"(a)())()"` | `["(a())()","(a)()()"]` |
| Ví dụ 3 | `")("` | `[""]` |

### Constraints (ràng buộc)

- `1 <= s.length <= 25`
- `s` chỉ gồm **chữ cái tiếng Anh viết thường** và các dấu ngoặc `'('`, `')'`.
- `s` có **nhiều nhất 20** dấu ngoặc.

Constraints rất nhỏ (25 ký tự, 20 ngoặc) là một **gợi ý quan trọng**: đề chấp nhận lời giải có độ phức tạp **hàm mũ** theo số ngoặc, miễn là ta không thử những thứ vô ích.

## 2. Hiểu đúng đề — bốn chi tiết dễ bỏ qua

Đánh số ký tự của ví dụ 1 để dễ nói chuyện:

| index | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| `s[i]` | `(` | `)` | `(` | `)` | `)` | `(` | `)` |

1. **"Xóa ít nhất"**: `"()())()"` dài 7. Chỉ cần xóa **1** ký tự là đã hợp lệ, nên **mọi** đáp án phải dài đúng 6. Chuỗi `"()()"` cũng hợp lệ nhưng phải xóa 3 ký tự → **không** được tính.
2. **"Tất cả, khác nhau"**: xóa `')'` ở index 1 → `"(())()"`; xóa `')'` ở index 3 **hoặc** index 4 đều ra `"()()()"`. Có 3 cách xóa nhưng chỉ ra **2 chuỗi khác nhau**, và mỗi chuỗi chỉ được xuất hiện **một lần** trong kết quả.
3. **Chữ cái**: đề chỉ cho xóa **ngoặc**. Chữ cái không ảnh hưởng tới tính hợp lệ — có thể coi chúng là "trong suốt" — và luôn được giữ nguyên.
4. **`")("` → `[""]`**: phải xóa cả hai ký tự, còn lại **chuỗi rỗng** `""`. Đáp án là một danh sách có **một phần tử là chuỗi rỗng**, **không phải** danh sách rỗng `[]`. Thực ra đáp án **luôn khác rỗng**: xóa hết mọi ngoặc thì chỉ còn chữ cái, và chuỗi chỉ có chữ cái luôn hợp lệ.

Cell dưới import vài thứ sẽ dùng xuyên suốt notebook và lưu 3 ví dụ của đề vào biến `examples` để test về sau.

In [1]:
from typing import List          # LeetCode dùng List[str] trong signature
from collections import deque    # hàng đợi cho BFS (học ở mục 7)
import random                    # sinh test ngẫu nhiên

examples = [
    ("()())()", ["(())()", "()()()"]),
    ("(a)())()", ["(a())()", "(a)()()"]),
    (")(", [""]),
]

for s, expected in examples:
    removed = len(s) - len(expected[0])
    print(f"s = {s!r:11} dài {len(s)} -> {expected}  (mỗi đáp án dài {len(expected[0])}, tức xóa {removed} ký tự)")

s = '()())()'   dài 7 -> ['(())()', '()()()']  (mỗi đáp án dài 6, tức xóa 1 ký tự)
s = '(a)())()'  dài 8 -> ['(a())()', '(a)()()']  (mỗi đáp án dài 7, tức xóa 1 ký tự)
s = ')('        dài 2 -> ['']  (mỗi đáp án dài 0, tức xóa 2 ký tự)


## 3. Kiến thức nền 1 — Kiểm tra chuỗi hợp lệ bằng biến đếm `balance`

Đây là phần **ôn lại** từ #20 Valid Parentheses và #1249 Minimum Remove to Make Valid Parentheses. Vì bài này chỉ có **một loại** ngoặc `()`, ta **không cần stack**: chỉ cần một số nguyên `balance` = "số dấu `'('` đang mở mà chưa được đóng".

- Gặp `'('` → `balance += 1` (mở thêm một ngoặc).
- Gặp `')'` → `balance -= 1` (đóng bớt một ngoặc).
- Gặp chữ cái → **không làm gì** (chữ cái "trong suốt").

Chuỗi hợp lệ **khi và chỉ khi** hai điều kiện cùng đúng:

1. **Trong lúc duyệt**, `balance` **không bao giờ âm**. Âm nghĩa là vừa gặp một `')'` mà chẳng có `'('` nào để ghép.
2. **Duyệt xong**, `balance == 0`: không còn `'('` nào bị bỏ lơ.

Ví dụ:

| Chuỗi | `balance` sau từng ký tự | Kết luận |
|---|---|---|
| `"(a)()"` | `1 → 1 → 0 → 1 → 0` | ✔ hợp lệ |
| `"())("` | `1 → 0 → -1` | ✘ âm ở index 2, dừng ngay |
| `"(()"` | `1 → 2 → 1` | ✘ kết thúc bằng 1 (thừa một `'('`) |

> **Cẩn thận:** nhiều bạn quen viết `balance += 1 if c == "(" else -1` từ các bài *chỉ có ngoặc* (như #32). Ở bài này có chữ cái, viết như vậy sẽ coi chữ cái là `')'` → sai. Phải dùng `elif c == ")"` (xem **Lỗi 1** ở mục 17).

In [2]:
def is_valid(t: str) -> bool:
    balance = 0
    for c in t:
        if c == "(":
            balance += 1
        elif c == ")":           # chữ cái không rơi vào nhánh nào -> bỏ qua
            balance -= 1
            if balance < 0:      # có ')' thừa -> hỏng ngay
                return False
    return balance == 0          # không được thừa '('

for t in ["()", "(a)()", "())(", "(()", "", "abc", ")(", "(a(b)c)"]:
    print(f"{t!r:10} -> {is_valid(t)}")

'()'       -> True
'(a)()'    -> True
'())('     -> False
'(()'      -> False
''         -> True
'abc'      -> True
')('       -> False
'(a(b)c)'  -> True


### Trace biến `balance`

Hàm `trace_balance` in giá trị `balance` **sau từng ký tự** (dạng `ký_tự:balance`) để thấy rõ thời điểm chuỗi "hỏng". Ta thử đúng 3 input của đề — cả ba đều **chưa** hợp lệ, và mỗi chuỗi hỏng theo một kiểu — cùng một chuỗi đáp án để so sánh.

In [3]:
def trace_balance(t: str) -> None:
    balance = 0
    steps = []
    for c in t:
        if c == "(":
            balance += 1
        elif c == ")":
            balance -= 1
        steps.append(f"{c}:{balance}")
        if balance < 0:
            print(f"{t!r:11} | {' '.join(steps)}  -> âm, KHÔNG hợp lệ")
            return
    verdict = "hợp lệ" if balance == 0 else f"thừa {balance} '(' -> KHÔNG hợp lệ"
    print(f"{t!r:11} | {' '.join(steps)}  -> {verdict}")

trace_balance("()())()")
trace_balance("(a)())()")
trace_balance(")(")
trace_balance("(())()")

'()())()'   | (:1 ):0 (:1 ):0 ):-1  -> âm, KHÔNG hợp lệ
'(a)())()'  | (:1 a:1 ):0 (:1 ):0 ):-1  -> âm, KHÔNG hợp lệ
')('        | ):-1  -> âm, KHÔNG hợp lệ
'(())()'    | (:1 (:2 ):1 ):0 (:1 ):0  -> hợp lệ


## 4. Kiến thức nền 2 — "Xóa ký tự", `set` và `sorted`

**Cắt chuỗi (slicing).** `t[a:b]` lấy các ký tự từ index `a` đến `b - 1`. Bỏ trống `a` nghĩa là "từ đầu", bỏ trống `b` nghĩa là "đến hết". Vậy muốn **xóa ký tự ở index `i`**, ta ghép phần trước và phần sau nó:

```python
t[:i] + t[i + 1:]      # mọi ký tự trừ t[i]
```

**Xóa nhiều ký tự = giữ lại một dãy con.** Các ký tự còn lại vẫn giữ **nguyên thứ tự**. Đó gọi là một **dãy con** (subsequence) của `s` — khác với **chuỗi con liên tiếp** (substring).

**`set` — tập hợp.** Một `set` không bao giờ chứa hai phần tử giống nhau. `add(x)` để thêm, `x in st` để hỏi "đã có chưa?" (trung bình `O(1)` với chuỗi ngắn). Ta sẽ dùng `set` để **bỏ trùng**, ví dụ hai cách xóa index 3 và 4 của `"()())"` cùng ra một chuỗi.

**`sorted` — sắp xếp.** Đề cho trả kết quả theo **thứ tự tùy ý**, nên khi test ta không so `a == b` trực tiếp mà so `sorted(a) == sorted(b)`.

In [4]:
t = "()())"
print("t[:3]        =", repr(t[:3]))       # 3 ký tự đầu
print("t[4:]        =", repr(t[4:]))       # từ index 4 đến hết
print("xóa index 3  =", repr(t[:3] + t[4:]))

children = [t[:i] + t[i + 1:] for i in range(len(t))]
print("Xóa từng ký tự một :", children)
print("Bỏ trùng bằng set  :", set(children))
print("Số chuỗi khác nhau :", len(set(children)), "(trong", len(children), "cách xóa)")

a = ["()()()", "(())()"]
b = ["(())()", "()()()"]
print("a == b:", a == b, "| sorted(a) == sorted(b):", sorted(a) == sorted(b))

t[:3]        = '()('
t[4:]        = ')'
xóa index 3  = '()()'
Xóa từng ký tự một : [')())', '(())', '()))', '()()', '()()']
Bỏ trùng bằng set  : {'(())', '()()', ')())', '()))'}
Số chuỗi khác nhau : 4 (trong 5 cách xóa)
a == b: False | sorted(a) == sorted(b): True


## 5. Kiến thức nền 3 — Bitmask: một số nguyên mô tả "giữ/xóa"

Với chuỗi dài `n`, mỗi ký tự có 2 lựa chọn: **giữ** hoặc **xóa** → tổng cộng `2^n` cách. Mẹo **bitmask** dùng một số nguyên `mask` từ `0` đến `2^n - 1` để mô tả một cách chọn: viết `mask` ở dạng **nhị phân**, **bit thứ `i` bằng 1 nghĩa là giữ `s[i]`**.

Ba phép toán cần biết:

| Biểu thức | Ý nghĩa | Ví dụ |
|---|---|---|
| `1 << n` | dịch số 1 sang trái `n` bit = `2^n` | `1 << 3 = 8` |
| `mask >> i` | dịch phải `i` bit (bỏ `i` bit cuối) | `6 >> 1 = 3` (`110 → 11`) |
| `(mask >> i) & 1` | đọc **bit thứ `i`** (0 hoặc 1) | `(6 >> 0) & 1 = 0`, `(6 >> 1) & 1 = 1` |

Bit được đánh số **từ phải sang trái**: bit 0 là chữ số nhị phân **cuối cùng**. Ví dụ `s = "a()"` (`n = 3`):

| mask | nhị phân (bit2 bit1 bit0) | giữ index | chuỗi giữ lại | hợp lệ? |
|:-:|:-:|:-:|:-:|:-:|
| 0 | `000` | – | `""` | ✔ |
| 1 | `001` | 0 | `"a"` | ✔ |
| 2 | `010` | 1 | `"("` | ✘ |
| 3 | `011` | 0, 1 | `"a("` | ✘ |
| 4 | `100` | 2 | `")"` | ✘ |
| 5 | `101` | 0, 2 | `"a)"` | ✘ |
| 6 | `110` | 1, 2 | `"()"` | ✔ |
| 7 | `111` | 0, 1, 2 | `"a()"` | ✔ |

`format(mask, "03b")` in `mask` ở dạng nhị phân đủ 3 chữ số (thêm số 0 ở đầu nếu thiếu).

In [5]:
print("1 << 3 =", 1 << 3, "  (= 2**3 cách chọn)")
print("6 ở dạng nhị phân:", format(6, "03b"),
      "| bit 0:", (6 >> 0) & 1, "| bit 1:", (6 >> 1) & 1, "| bit 2:", (6 >> 2) & 1)

s = "a()"
n = len(s)
for mask in range(1 << n):
    kept = [i for i in range(n) if (mask >> i) & 1]
    t = "".join(s[i] for i in kept)
    print(f"mask={mask} ({format(mask, '03b')}) giữ index {str(kept):9} -> {t!r:6} hợp lệ? {is_valid(t)}")

1 << 3 = 8   (= 2**3 cách chọn)
6 ở dạng nhị phân: 110 | bit 0: 0 | bit 1: 1 | bit 2: 1
mask=0 (000) giữ index []        -> ''     hợp lệ? True
mask=1 (001) giữ index [0]       -> 'a'    hợp lệ? True
mask=2 (010) giữ index [1]       -> '('    hợp lệ? False
mask=3 (011) giữ index [0, 1]    -> 'a('   hợp lệ? False
mask=4 (100) giữ index [2]       -> ')'    hợp lệ? False
mask=5 (101) giữ index [0, 2]    -> 'a)'   hợp lệ? False
mask=6 (110) giữ index [1, 2]    -> '()'   hợp lệ? True
mask=7 (111) giữ index [0, 1, 2] -> 'a()'  hợp lệ? True


## 6. Cách 1 — Brute force bằng bitmask (dùng làm "trọng tài")

**Ý tưởng tự nhiên nhất:** thử **mọi** cách giữ/xóa ký tự, giữ lại những chuỗi hợp lệ **dài nhất**.

Vì sao "dài nhất" chính là "xóa ít nhất"? Vì `len(chuỗi còn lại) = n - (số ký tự đã xóa)`. Chuỗi còn lại càng dài thì số ký tự xóa càng ít.

Còn chuyện "chỉ được xóa ngoặc, không xóa chữ cái"? Ta **không cần** xử lý riêng: nếu một mask xóa một chữ cái mà vẫn hợp lệ, thì mask giống hệt nhưng **giữ** chữ cái đó cũng hợp lệ (chữ cái trong suốt) và dài hơn 1. Nên mask xóa chữ cái **không bao giờ** đạt độ dài lớn nhất.

Thuật toán:

1. Duyệt `mask` từ `0` đến `2^n - 1`, dựng chuỗi `t` gồm các ký tự có bit bằng 1.
2. Nếu `t` hợp lệ: dài hơn kỷ lục → **reset** tập đáp án thành `{t}`; bằng kỷ lục → **thêm** `t` vào tập.
3. Dùng `set` nên hai mask cho cùng một chuỗi (như xóa index 3 hay 4 của ví dụ 1) chỉ được đếm một lần.

**Độ phức tạp:** `2^n` mask, mỗi mask dựng và kiểm tra chuỗi mất `O(n)` → **Time `O(2^n · n)`**, **Space `O(n)`** cho chuỗi đang xét (chưa kể tập đáp án).

In [6]:
def remove_invalid_bruteforce(s: str) -> List[str]:
    n = len(s)
    best_len = -1
    best = set()
    for mask in range(1 << n):
        t = "".join(s[i] for i in range(n) if (mask >> i) & 1)
        if not is_valid(t):
            continue
        if len(t) > best_len:          # kỷ lục mới -> bỏ các chuỗi ngắn hơn
            best_len = len(t)
            best = {t}
        elif len(t) == best_len:       # bằng kỷ lục -> thêm vào tập
            best.add(t)
    return list(best)

for s, expected in examples:
    got = remove_invalid_bruteforce(s)
    print(f"{s!r:11} -> {sorted(got)}")
    assert sorted(got) == sorted(expected)
print("Brute force khớp cả 3 ví dụ của đề")

'()())()'   -> ['(())()', '()()()']
'(a)())()'  -> ['(a())()', '(a)()()']
')('        -> ['']
Brute force khớp cả 3 ví dụ của đề


### Nút thắt của brute force: thử quá nhiều cách xóa vô ích

Cell dưới đếm, trong `2^7 = 128` mask của `"()())()"`, có bao nhiêu mask cho chuỗi hợp lệ, **chia theo số ký tự đã xóa**. Ta chỉ cần những mask xóa **đúng 1** ký tự (có 3 mask như vậy: xóa index 1, 3 hoặc 4), vậy mà brute force vẫn dựng và kiểm tra **cả 128** chuỗi. Kết quả bên dưới: trong 23 mask cho chuỗi hợp lệ, có tới 20 mask xóa 3, 5, thậm chí cả 7 ký tự — toàn là việc vô ích.

Tệ hơn, với `n = 25` (giới hạn của đề) có hơn **33 triệu** mask; mỗi mask lại dựng một chuỗi dài tới 25 ký tự — Python phải chạy cỡ vài phút. Brute force chỉ còn giá trị làm **trọng tài** trên chuỗi ngắn.

`Counter` (trong `collections`) là một `dict` chuyên đếm: `cnt[key] += 1` mà không cần khởi tạo trước.

In [7]:
from collections import Counter

s = "()())()"
n = len(s)
valid_by_deleted = Counter()
for mask in range(1 << n):
    t = "".join(s[i] for i in range(n) if (mask >> i) & 1)
    if is_valid(t):
        valid_by_deleted[n - len(t)] += 1

print("Tổng số mask phải thử:", 1 << n)
print("Mask cho chuỗi hợp lệ, theo số ký tự đã xóa:", dict(sorted(valid_by_deleted.items())))
print(f"Với n = 25 phải thử {1 << 25:,} mask")

Tổng số mask phải thử: 128
Mask cho chuỗi hợp lệ, theo số ký tự đã xóa: {1: 3, 3: 11, 5: 8, 7: 1}
Với n = 25 phải thử 33,554,432 mask


## 7. Kiến thức nền 4 — BFS (tìm kiếm theo chiều rộng) và `deque`

Brute force vướng ở chỗ: nó thử các cách xóa **không theo thứ tự nào**. Nếu ta thử **xóa 0 ký tự**, rồi **xóa 1**, rồi **xóa 2**... và **dừng ngay** khi gặp chuỗi hợp lệ, thì chắc chắn đó là số lần xóa ít nhất. Cách duyệt "lan dần theo khoảng cách" này chính là **BFS**.

**Hình dung:** ném hòn đá xuống mặt hồ, sóng lan ra từng vòng: vòng 1, vòng 2, vòng 3... BFS thăm các **trạng thái** (state) theo đúng thứ tự **khoảng cách** tới điểm xuất phát: khoảng cách 0, rồi 1, rồi 2...

Ở bài này:

- **Trạng thái** = một chuỗi.
- **Đi một bước** = xóa **một** ký tự ngoặc.
- **Khoảng cách** = số ký tự đã xóa.

**`deque`** (double-ended queue — hàng đợi hai đầu) là công cụ chuẩn cho BFS:

- `q.append(x)`: thêm `x` vào **cuối** hàng.
- `q.popleft()`: lấy phần tử ở **đầu** hàng ra → **vào trước ra trước** (FIFO — First In First Out), như xếp hàng mua vé.

Vì sao không dùng `list.pop(0)`? Vì `list` phải dời toàn bộ phần tử còn lại lên một ô → `O(len)` mỗi lần; `deque.popleft()` chỉ `O(1)`.

Cell dưới: thí nghiệm `deque`, rồi chạy một BFS nhỏ trên chuỗi `"abc"` (mỗi bước xóa 1 ký tự bất kỳ). Quan sát: mọi chuỗi xóa 1 ký tự được thăm **trước** mọi chuỗi xóa 2 ký tự.

In [8]:
q = deque()
q.append("A"); q.append("B"); q.append("C")
print(q)
print("popleft ->", q.popleft(), "| còn lại:", q)

# BFS nhỏ: từ "abc", mỗi bước xóa 1 ký tự bất kỳ
start = "abc"
queue = deque([(start, 0)])          # (chuỗi, số ký tự đã xóa)
visited = {start}
while queue:
    t, dist = queue.popleft()
    print(f"thăm {t!r:6} | đã xóa {dist} ký tự")
    for i in range(len(t)):
        child = t[:i] + t[i + 1:]
        if child not in visited:     # chưa từng gặp -> xếp hàng
            visited.add(child)
            queue.append((child, dist + 1))

deque(['A', 'B', 'C'])
popleft -> A | còn lại: deque(['B', 'C'])
thăm 'abc'  | đã xóa 0 ký tự
thăm 'bc'   | đã xóa 1 ký tự
thăm 'ac'   | đã xóa 1 ký tự
thăm 'ab'   | đã xóa 1 ký tự
thăm 'c'    | đã xóa 2 ký tự
thăm 'b'    | đã xóa 2 ký tự
thăm 'a'    | đã xóa 2 ký tự
thăm ''     | đã xóa 3 ký tự


### BFS theo tầng (level-by-level) và vì sao cần `visited`

Thay vì lưu khoảng cách kèm từng chuỗi, ta có thể xử lý **trọn một tầng** mỗi lần: `level` là danh sách các chuỗi đã xóa đúng `k` ký tự; từ `level` sinh ra tầng `k + 1`.

**Vấn đề trùng lặp.** Với `"())"`:

- Tầng 0: `["())"]`
- Tầng 1: xóa index 0 → `"))"`, xóa index 1 → `"()"`, xóa index 2 → `"()"` ← **trùng**!

Trùng lặp còn nhân lên theo tầng: xóa index 1 rồi index 3, hay xóa index 3 rồi index 1 (thứ tự ngược lại), đều ra cùng một chuỗi. Nếu không lọc, tầng `k` của một chuỗi `p` ngoặc có tới `p · (p-1) · ... · (p-k+1)` phần tử (đếm cả thứ tự xóa), trong khi số **tập** vị trí bị xóa chỉ là `C(p, k)`. Ví dụ `p = 10, k = 3`: `720` so với `120`.

Tập **`visited`** ghi nhớ mọi chuỗi đã từng sinh ra; chuỗi nào đã có thì **không** thêm vào tầng sau nữa. Cell dưới so sánh kích thước các tầng của `"()())()"` khi có và không có `visited`.

In [9]:
def level_sizes(s: str, use_visited: bool, max_level: int) -> List[int]:
    level = [s]
    visited = {s}
    sizes = [1]
    for _ in range(max_level):
        next_level = []
        for t in level:
            for i, c in enumerate(t):
                if c not in "()":
                    continue
                child = t[:i] + t[i + 1:]
                if use_visited:
                    if child in visited:
                        continue
                    visited.add(child)
                next_level.append(child)
        level = next_level
        sizes.append(len(level))
    return sizes

t = "())"
print("Con của", repr(t), ":", [t[:i] + t[i + 1:] for i in range(len(t))])
print("Kích thước tầng 0..3 của '()())()' KHÔNG visited:", level_sizes("()())()", False, 3))
print("Kích thước tầng 0..3 của '()())()' CÓ visited   :", level_sizes("()())()", True, 3))

Con của '())' : ['))', '()', '()']
Kích thước tầng 0..3 của '()())()' KHÔNG visited: [1, 7, 42, 210]
Kích thước tầng 0..3 của '()())()' CÓ visited   : [1, 6, 13, 13]


## 8. Cách 2 — BFS theo tầng cho bài 301

Cell trên cho thấy sức mạnh của `visited`: tầng 3 của `"()())()"` thu từ **210** phần tử xuống chỉ còn **13** chuỗi khác nhau. Giờ ghép lại thành thuật toán hoàn chỉnh:

```text
level   = [s]           # tầng 0: chưa xóa gì
visited = {s}
lặp:
    valid = các chuỗi hợp lệ trong level
    nếu valid khác rỗng: trả về valid          # DỪNG ở tầng đầu tiên có đáp án
    next_level = []
    với mỗi t trong level, với mỗi index i mà t[i] là NGOẶC:
        child = t[:i] + t[i+1:]
        nếu child chưa có trong visited: thêm vào visited và next_level
    level = next_level
```

**Vì sao BFS đảm bảo "ít nhất"?**

- Tầng `k` chứa **đúng tất cả** các chuỗi khác nhau thu được bằng cách xóa `k` ngoặc. (Mọi cách xóa `k + 1` ngoặc đều là "một cách xóa `k` ngoặc rồi xóa thêm 1", nên tầng sau được sinh **đầy đủ** từ tầng trước.)
- Ta xét tầng 0, 1, 2... **theo thứ tự**. Nếu các tầng `0..k-1` không có chuỗi hợp lệ nào mà tầng `k` có, thì không thể làm được với ít hơn `k` lần xóa → **`k` là ít nhất**.
- Ta lấy **trọn** tầng `k` (không trả về ngay chuỗi hợp lệ đầu tiên) → không sót đáp án; và **không** đi tiếp xuống tầng `k + 1` → không lẫn đáp án xóa nhiều hơn.

**Vai trò của `visited`:** mỗi bước xóa đúng 1 ký tự nên các chuỗi ở hai tầng khác nhau luôn **khác độ dài** — chúng không bao giờ trùng nhau. Vậy `visited` thực chất đang lọc trùng **trong cùng một tầng**, nhờ đó mỗi tầng có không quá `C(p, k)` chuỗi và đáp án trả về tự động không trùng.

**Chỉ xóa ngoặc:** xóa chữ cái không bao giờ giúp chuỗi hợp lệ hơn, chỉ làm tầng phình to.

In [10]:
def remove_invalid_bfs(s: str) -> List[str]:
    level = [s]
    visited = {s}
    while True:
        valid = [t for t in level if is_valid(t)]
        if valid:                          # tầng đầu tiên có chuỗi hợp lệ
            return valid
        next_level = []
        for t in level:
            for i, c in enumerate(t):
                if c not in "()":          # chỉ xóa ngoặc
                    continue
                child = t[:i] + t[i + 1:]
                if child not in visited:
                    visited.add(child)
                    next_level.append(child)
        level = next_level

for s, expected in examples:
    got = remove_invalid_bfs(s)
    print(f"{s!r:11} -> {got}")
    assert sorted(got) == sorted(expected)
print("BFS khớp cả 3 ví dụ của đề")

'()())()'   -> ['(())()', '()()()']
'(a)())()'  -> ['(a())()', '(a)()()']
')('        -> ['']
BFS khớp cả 3 ví dụ của đề


### Trace BFS cho `"()())()"`

Tầng 0 chỉ có chính `s`, không hợp lệ. Sinh tầng 1 bằng cách xóa lần lượt từng ngoặc (cả 7 ký tự đều là ngoặc):

| xóa index | ký tự bị xóa | chuỗi con | `visited`? | hợp lệ? |
|:-:|:-:|---|---|:-:|
| 0 | `(` | `")())()"` | mới → thêm | ✘ |
| 1 | `)` | `"(())()"` | mới → thêm | ✔ |
| 2 | `(` | `"()))()"` | mới → thêm | ✘ |
| 3 | `)` | `"()()()"` | mới → thêm | ✔ |
| 4 | `)` | `"()()()"` | **đã có** → bỏ | – |
| 5 | `(` | `"()()))"` | mới → thêm | ✘ |
| 6 | `)` | `"()())("` | mới → thêm | ✘ |

Tầng 1 có 6 chuỗi, trong đó 2 chuỗi hợp lệ → **dừng**, số lần xóa ít nhất là 1, đáp án `["(())()", "()()()"]`.

Hàm `trace_bfs` dưới in mỗi tầng (✔ = hợp lệ, ✘ = không) và số chuỗi con sinh ra / bị loại vì đã thăm. Ta chạy thêm `")("` để thấy BFS phải xuống tới **tầng 2** và đáp án là chuỗi rỗng.

In [11]:
def trace_bfs(s: str) -> None:
    level = [s]
    visited = {s}
    k = 0
    while True:
        marks = "  ".join(("✔" if is_valid(t) else "✘") + repr(t) for t in level)
        print(f"Tầng {k} ({len(level)} chuỗi): {marks}")
        valid = [t for t in level if is_valid(t)]
        if valid:
            print(f"  -> DỪNG ở tầng {k}: xóa ít nhất {k} ký tự, đáp án = {valid}")
            return
        next_level, made, dup = [], 0, 0
        for t in level:
            for i, c in enumerate(t):
                if c in "()":
                    child = t[:i] + t[i + 1:]
                    made += 1
                    if child in visited:
                        dup += 1
                    else:
                        visited.add(child)
                        next_level.append(child)
        print(f"  -> chưa có chuỗi hợp lệ: sinh {made} chuỗi con, bỏ {dup} chuỗi đã thăm")
        level = next_level
        k += 1

trace_bfs("()())()")
print()
trace_bfs(")(")

Tầng 0 (1 chuỗi): ✘'()())()'
  -> chưa có chuỗi hợp lệ: sinh 7 chuỗi con, bỏ 1 chuỗi đã thăm
Tầng 1 (6 chuỗi): ✘')())()'  ✔'(())()'  ✘'()))()'  ✔'()()()'  ✘'()()))'  ✘'()())('
  -> DỪNG ở tầng 1: xóa ít nhất 1 ký tự, đáp án = ['(())()', '()()()']

Tầng 0 (1 chuỗi): ✘')('
  -> chưa có chuỗi hợp lệ: sinh 2 chuỗi con, bỏ 0 chuỗi đã thăm
Tầng 1 (2 chuỗi): ✘'('  ✘')'
  -> chưa có chuỗi hợp lệ: sinh 2 chuỗi con, bỏ 1 chuỗi đã thăm
Tầng 2 (1 chuỗi): ✔''
  -> DỪNG ở tầng 2: xóa ít nhất 2 ký tự, đáp án = ['']


### Độ phức tạp của BFS và nút thắt còn lại

Gọi `k` là số lần xóa ít nhất. Tầng `j` có tối đa `C(p, j)` chuỗi; mỗi chuỗi sinh tối đa `p` con, mỗi con tốn `O(n)` để cắt ghép và tra `visited`.

- **Time:** `O(p · n · (C(p,0) + C(p,1) + ... + C(p,k)))`, xấu nhất cỡ `O(p · n · 2^p)`.
- **Space:** `visited` giữ mọi chuỗi đã sinh → cùng bậc với số chuỗi đó, nhân `O(n)` cho độ dài mỗi chuỗi.

> **Ghi chú — bản BFS "một hàng đợi + cờ `found`" hay gặp trên mạng:** dùng một `deque` duy nhất, gặp chuỗi hợp lệ thì bật `found = True` và thôi sinh con, nhưng vẫn lấy nốt các chuỗi còn trong hàng. Bản đó đúng nhờ một tính chất **chẵn lẻ**: chuỗi hợp lệ luôn có số ngoặc chẵn, mà hai tầng liền nhau có số ngoặc chênh nhau 1 → các chuỗi "tầng sau" lỡ nằm trong hàng không thể hợp lệ. Bản theo tầng ở trên dễ hiểu hơn nên ta dùng nó.

**Nút thắt:** BFS **mù quáng** — nó không biết nên xóa loại ngoặc nào. Với `"()())()"` ta chỉ cần xóa một `')'`, vậy mà tầng 1 vẫn có 3 chuỗi do xóa `'('` (index 0, 2, 5), toàn là chuỗi vô dụng. Khi cần xóa nhiều ký tự, phần vô dụng này phình rất to. Cell dưới xét chuỗi `')()(()()))(a)((()(()'` (20 ký tự, 19 ngoặc — vẫn trong giới hạn đề; như mục 9 sẽ cho thấy, mọi đáp án phải xóa đúng 3 `'('` và 2 `')'`) và thống kê **tầng cuối cùng** theo kiểu "đã xóa bao nhiêu `'('`, bao nhiêu `')'`".

In [12]:
s = ")()(()()))(a)((()(()"
level = [s]
visited = {s}
sizes = [1]
while not any(is_valid(t) for t in level):
    next_level = []
    for t in level:
        for i, c in enumerate(t):
            if c in "()":
                child = t[:i] + t[i + 1:]
                if child not in visited:
                    visited.add(child)
                    next_level.append(child)
    level = next_level
    sizes.append(len(level))

print("Kích thước từng tầng:", sizes, "| tổng số chuỗi đã sinh:", sum(sizes))
composition = Counter((s.count("(") - t.count("("), s.count(")") - t.count(")")) for t in level)
print("Tầng cuối, theo (số '(' đã xóa, số ')' đã xóa):", dict(composition))
print("Số chuỗi hợp lệ ở tầng cuối:", sum(is_valid(t) for t in level))

Kích thước từng tầng: [1, 13, 77, 279, 702, 1320] | tổng số chuỗi đã sinh: 2392
Tầng cuối, theo (số '(' đã xóa, số ')' đã xóa): {(3, 2): 388, (2, 3): 376, (1, 4): 212, (4, 1): 228, (0, 5): 56, (5, 0): 60}
Số chuỗi hợp lệ ở tầng cuối: 6


## 9. Mấu chốt: đếm trước số ngoặc **bắt buộc** phải xóa — `left_rem`, `right_rem`

Thí nghiệm ở cell trên: tầng cuối của BFS có 1320 chuỗi, nhưng chỉ **388** chuỗi xóa đúng "3 `'('` + 2 `')'`"; **932** chuỗi còn lại xóa sai loại ngoặc nên chắc chắn hỏng — và cuối cùng chỉ có 6 đáp án.

Nếu biết trước "phải xóa đúng **mấy** `'('` và **mấy** `')'`", ta khỏi thử bừa như BFS. Và điều này đếm được bằng **một lượt quét**, giống hệt cách làm #921 / #1249:

```text
open_count = 0, right_rem = 0
gặp '('  -> open_count += 1                    # thêm một '(' đang chờ ghép
gặp ')'  -> nếu open_count > 0: open_count -= 1  # ghép với một '(' đang chờ
            ngược lại:          right_rem += 1   # ')' này không có ai ghép -> chắc chắn thừa
gặp chữ  -> bỏ qua
cuối cùng: left_rem = open_count                # các '(' không bao giờ được đóng
```

Ví dụ `"()())()"`: `( ) ( )` ghép xong hết, tới `')'` ở index 4 thì `open_count = 0` → `right_rem = 1`; phần `()` còn lại ghép được. Kết quả `left_rem = 0, right_rem = 1`.

**Vì sao đây là số TỐI THIỂU?**

- *Phải xóa ít nhất `right_rem` dấu `')'`:* khi `right_rem` tăng lên thành `r` tại index `i`, đoạn đầu `s[0..i]` có số `')'` nhiều hơn số `'('` đúng `r`. Chuỗi kết quả hợp lệ thì mọi đoạn đầu phải có `')'` **không nhiều hơn** `'('`. Xóa `'('` chỉ làm tệ thêm, nên **bắt buộc** xóa ít nhất `r` dấu `')'` trong đoạn đó.
- *Phải xóa ít nhất `left_rem` dấu `'('`:* mỗi cặp ghép được góp một `'('` và một `')'`, nên trong `s` số `'('` nhiều hơn số `')'` đúng `left_rem - right_rem`. Chuỗi kết quả phải có số `'('` **bằng** số `')'`, do đó `(số '(' bị xóa) = (số ')' bị xóa) + left_rem - right_rem ≥ right_rem + left_rem - right_rem = left_rem`. Ví dụ `"())((a)("`: `s` có 5 `'('`, 3 `')'`; nếu xóa 1 `')'` thì phải xóa `1 + 2 - 1 = 2` dấu `'('`.
- *Đạt được:* xóa đúng những ngoặc "không có ai ghép" vừa đếm thì được một chuỗi hợp lệ (đó chính là lời giải #1249 — Bài tập 5).

Vậy **số lần xóa ít nhất = `left_rem + right_rem`**, và mọi đáp án đều xóa **đúng `left_rem` dấu `'('` và đúng `right_rem` dấu `')'`** — xóa thêm bất kỳ ngoặc nào cũng làm tổng vượt mức tối thiểu.

Cell dưới cài `count_removals` và một bản `trace_count_removals` in từng bước cho `"())((a)("`.

In [13]:
def count_removals(s: str):
    open_count = 0
    right_rem = 0
    for c in s:
        if c == "(":
            open_count += 1
        elif c == ")":
            if open_count > 0:
                open_count -= 1      # ghép cặp
            else:
                right_rem += 1       # ')' không có ai ghép
    left_rem = open_count            # '(' không ai đóng
    return left_rem, right_rem

for s in ["()())()", "(a)())()", ")(", "(((", "a)b(c)d", "())((a)("]:
    l, r = count_removals(s)
    print(f"{s!r:11} -> left_rem={l}, right_rem={r}, xóa ít nhất {l + r} ký tự")


def trace_count_removals(s: str) -> None:
    open_count = right_rem = 0
    for i, c in enumerate(s):
        if c == "(":
            open_count += 1
            action = "mở thêm"
        elif c == ")" and open_count > 0:
            open_count -= 1
            action = "ghép cặp"
        elif c == ")":
            right_rem += 1
            action = "KHÔNG ai ghép -> right_rem += 1"
        else:
            action = "chữ cái, bỏ qua"
        print(f"  i={i} {c!r}: {action:32} open_count={open_count}, right_rem={right_rem}")
    print(f"  => left_rem = open_count = {open_count}, right_rem = {right_rem}")

print()
print("Trace '())((a)(':")
trace_count_removals("())((a)(")

'()())()'   -> left_rem=0, right_rem=1, xóa ít nhất 1 ký tự
'(a)())()'  -> left_rem=0, right_rem=1, xóa ít nhất 1 ký tự
')('        -> left_rem=1, right_rem=1, xóa ít nhất 2 ký tự
'((('       -> left_rem=3, right_rem=0, xóa ít nhất 3 ký tự
'a)b(c)d'   -> left_rem=0, right_rem=1, xóa ít nhất 1 ký tự
'())((a)('  -> left_rem=2, right_rem=1, xóa ít nhất 3 ký tự

Trace '())((a)(':
  i=0 '(': mở thêm                          open_count=1, right_rem=0
  i=1 ')': ghép cặp                         open_count=0, right_rem=0
  i=2 ')': KHÔNG ai ghép -> right_rem += 1  open_count=0, right_rem=1
  i=3 '(': mở thêm                          open_count=1, right_rem=1
  i=4 '(': mở thêm                          open_count=2, right_rem=1
  i=5 'a': chữ cái, bỏ qua                  open_count=2, right_rem=1
  i=6 ')': ghép cặp                         open_count=1, right_rem=1
  i=7 '(': mở thêm                          open_count=2, right_rem=1
  => left_rem = open_count = 2, right_rem = 1


## 10. Kiến thức nền 5 — Đệ quy (recursion) và quay lui (backtracking)

**Đệ quy** là khi một hàm **gọi lại chính nó** để giải một bài nhỏ hơn. Luôn cần một **điểm dừng** (base case), nếu không hàm sẽ gọi mãi.

**Quay lui (backtracking)** là cách dựng lời giải **từng bước**, lưu phần đã dựng trong một danh sách `path`. Ở mỗi bước ta **thử từng lựa chọn** theo nhịp 3 nhịp:

1. **Chọn** — `path.append(...)`
2. **Đi sâu** — gọi đệ quy cho bước tiếp theo
3. **Hoàn tác** — `path.pop()` để trả `path` về như cũ, sẵn sàng cho lựa chọn kế tiếp.

**Ví dụ mini:** liệt kê mọi dãy con của `"abc"`. Mỗi ký tự có 2 lựa chọn **bỏ** / **giữ** → `2^3 = 8` dãy con. Cây quyết định (mỗi tầng là một ký tự, nhánh trái = bỏ, nhánh phải = giữ):

```text
                         ""                         <- bắt đầu, path rỗng
            bỏ 'a' /            \ giữ 'a'
                 ""                "a"
          bỏ 'b'/   \giữ 'b'   /        \
             ""      "b"     "a"        "ab"
            /  \    /  \    /   \      /    \
          ""  "c" "b" "bc" "a" "ac"  "ab"  "abc"    <- 8 lá = 8 dãy con
```

Mỗi **lá** (hết chuỗi) là một kết quả. Hàm `trace_subsequences` in từng lời gọi với **thụt lề theo độ sâu** để bạn "nhìn thấy" call stack (ngăn xếp lời gọi): dòng thụt sâu hơn là lời gọi con của dòng phía trên nó.

In [14]:
def all_subsequences(s: str) -> List[str]:
    result = []
    path = []

    def dfs(i):
        if i == len(s):                 # base case: đã quyết định xong mọi ký tự
            result.append("".join(path))
            return
        dfs(i + 1)                      # lựa chọn 1: BỎ s[i]
        path.append(s[i])               # lựa chọn 2: GIỮ s[i]  (chọn)
        dfs(i + 1)                      #                       (đi sâu)
        path.pop()                      #                       (hoàn tác)

    dfs(0)
    return result

print(all_subsequences("abc"))


def trace_subsequences(s: str) -> None:
    path = []

    def dfs(i):
        indent = "    " * i
        if i == len(s):
            print(f"{indent}lá: {''.join(path)!r}")
            return
        print(f"{indent}i={i}: bỏ {s[i]!r}")
        dfs(i + 1)
        print(f"{indent}i={i}: giữ {s[i]!r}")
        path.append(s[i])
        dfs(i + 1)
        path.pop()

    dfs(0)

print()
trace_subsequences("ab")

['', 'c', 'b', 'bc', 'a', 'ac', 'ab', 'abc']

i=0: bỏ 'a'
    i=1: bỏ 'b'
        lá: ''
    i=1: giữ 'b'
        lá: 'b'
i=0: giữ 'a'
    i=1: bỏ 'b'
        lá: 'a'
    i=1: giữ 'b'
        lá: 'ab'


## 11. Cách 3 — DFS/backtracking xóa đúng `left_rem` `'('` và `right_rem` `')'` (cách chính)

Giờ ghép hai ý: **duyệt giữ/xóa từng ký tự** như cây quyết định ở trên, nhưng **có hạn mức** (quota) xóa lấy từ `count_removals`. Hàm đệ quy nhớ 4 thông tin:

| Tham số | Ý nghĩa |
|---|---|
| `i` | đang quyết định ký tự `s[i]` |
| `open_count` | số `'('` đang mở trong `path` (chính là `balance` của `path`) |
| `del_left` | số `'('` **còn phải** xóa |
| `del_right` | số `')'` **còn phải** xóa |

Tại ký tự `c = s[i]`:

- **Chữ cái:** luôn **giữ**.
- **`'('`:** nhánh **XÓA** (chỉ khi `del_left > 0`) và nhánh **GIỮ** (`open_count + 1`).
- **`')'`:** nhánh **XÓA** (chỉ khi `del_right > 0`) và nhánh **GIỮ** — nhưng **chỉ khi `open_count > 0`**.

**Điểm dừng:** `i == n`. Nếu `del_left == del_right == 0` và `open_count == 0` thì `path` là một đáp án.

**Hai nhát cắt tỉa (pruning)** — bỏ sớm những nhánh chắc chắn không ra đáp án:

- **(P1) Hạn mức:** không bao giờ xóa quá `left_rem` / `right_rem` → mọi chuỗi sinh ra đều xóa **đúng** số tối thiểu, không cần "so độ dài" như brute force.
- **(P2) Không giữ `')'` khi `open_count == 0`:** giữ nó thì `balance` của `path` âm → `path` đã hỏng vĩnh viễn (xóa ký tự phía sau không cứu được đoạn đầu). Nhờ P2, `open_count` **không bao giờ âm**, nên điều kiện ở lá cộng với P2 đúng bằng hai điều kiện hợp lệ ở mục 3.

**Phiên bản A** dưới đây dùng `set` để bỏ trùng (vì xóa index 3 hay 4 của `"()())()"` vẫn ra cùng chuỗi). Độ sâu đệ quy chỉ tối đa `n = 25` nên không lo giới hạn đệ quy của Python.

In [15]:
def remove_invalid_dfs_set(s: str) -> List[str]:
    left_rem, right_rem = count_removals(s)
    n = len(s)
    result = set()
    path = []

    def dfs(i, open_count, del_left, del_right):
        if i == n:
            if del_left == 0 and del_right == 0 and open_count == 0:
                result.add("".join(path))
            return
        c = s[i]
        if c == "(":
            if del_left > 0:                                   # XÓA '(' (P1)
                dfs(i + 1, open_count, del_left - 1, del_right)
            path.append(c)                                     # GIỮ '('
            dfs(i + 1, open_count + 1, del_left, del_right)
            path.pop()
        elif c == ")":
            if del_right > 0:                                  # XÓA ')' (P1)
                dfs(i + 1, open_count, del_left, del_right - 1)
            if open_count > 0:                                 # GIỮ ')' (P2)
                path.append(c)
                dfs(i + 1, open_count - 1, del_left, del_right)
                path.pop()
        else:                                                  # chữ cái: luôn giữ
            path.append(c)
            dfs(i + 1, open_count, del_left, del_right)
            path.pop()

    dfs(0, 0, left_rem, right_rem)
    return list(result)

for s, expected in examples:
    got = remove_invalid_dfs_set(s)
    print(f"{s!r:11} -> {sorted(got)}")
    assert sorted(got) == sorted(expected)
print("DFS (bản set) khớp cả 3 ví dụ của đề")

'()())()'   -> ['(())()', '()()()']
'(a)())()'  -> ['(a())()', '(a)()()']
')('        -> ['']
DFS (bản set) khớp cả 3 ví dụ của đề


### Bỏ trùng không cần `set`: quy tắc "chỉ xóa phần đầu của dãy ngoặc giống nhau"

**Trùng sinh ra từ đâu?** Trong `"()())"`, hai ký tự `'))'` ở index 3, 4 đứng **liền nhau**. Xóa cái này hay cái kia, chuỗi còn lại như nhau. Tổng quát: trong một **dãy liền nhau các ngoặc giống hệt nhau**, muốn xóa `x` ký tự thì xóa *những ký tự nào* trong dãy cũng ra cùng kết quả — chỉ có **số lượng** `x` là quan trọng.

**Quy tắc:** trong mỗi dãy như vậy, ta **chỉ xóa `x` ký tự đầu tiên**. Nói cách khác: *nếu ký tự ngay trước `s[i]` giống hệt `s[i]` và đã được **GIỮ**, thì **không được XÓA** `s[i]`* — vì "giữ cái trước, xóa cái sau" cho đúng chuỗi của "xóa cái trước, giữ cái sau", mà nhánh đó đã được duyệt rồi.

Cài đặt: thêm tham số `prev_kept` (ký tự trước có được giữ không) và tính

```python
can_delete = not (i > 0 and s[i - 1] == c and prev_kept)
```

(Chữ cái luôn được giữ và không bao giờ bằng ngoặc `c`, nên chúng đóng vai "cọc mốc" ngăn cách các dãy.)

Quy tắc này chỉ loại **trùng trong cùng một dãy**. Liệu có trùng kiểu khác — hai cách xóa ở những chỗ cách xa nhau mà vẫn ra cùng chuỗi? Mục 15 sẽ chứng minh là **không**, nhờ một tính chất đặc biệt của cách xóa tối thiểu. Cell dưới cài **phiên bản B** với cờ `skip_duplicates`: tắt cờ thì thấy rõ trùng lặp, bật cờ thì hết trùng.

In [16]:
def remove_invalid_dfs(s: str, skip_duplicates: bool = True) -> List[str]:
    left_rem, right_rem = count_removals(s)
    n = len(s)
    result = []                       # list, KHÔNG dùng set
    path = []

    def dfs(i, open_count, del_left, del_right, prev_kept):
        if i == n:
            if del_left == 0 and del_right == 0 and open_count == 0:
                result.append("".join(path))
            return
        c = s[i]
        can_delete = not (skip_duplicates and i > 0 and s[i - 1] == c and prev_kept)
        if c == "(":
            if del_left > 0 and can_delete:
                dfs(i + 1, open_count, del_left - 1, del_right, False)
            path.append(c)
            dfs(i + 1, open_count + 1, del_left, del_right, True)
            path.pop()
        elif c == ")":
            if del_right > 0 and can_delete:
                dfs(i + 1, open_count, del_left, del_right - 1, False)
            if open_count > 0:
                path.append(c)
                dfs(i + 1, open_count - 1, del_left, del_right, True)
                path.pop()
        else:
            path.append(c)
            dfs(i + 1, open_count, del_left, del_right, True)
            path.pop()

    dfs(0, 0, left_rem, right_rem, False)
    return result

for s in ["()())()", "(()))))", "((a)"]:
    raw = remove_invalid_dfs(s, skip_duplicates=False)
    clean = remove_invalid_dfs(s)
    print(f"{s!r:10} không bỏ trùng: {raw}")
    print(f"{'':10} có bỏ trùng  : {clean}")
    assert len(clean) == len(set(clean)) and sorted(clean) == sorted(set(raw))
for s, expected in examples:
    assert sorted(remove_invalid_dfs(s)) == sorted(expected)
print("DFS (bản bỏ trùng) khớp cả 3 ví dụ của đề, không sinh trùng")

'()())()'  không bỏ trùng: ['(())()', '()()()', '()()()']
           có bỏ trùng  : ['(())()', '()()()']
'(()))))'  không bỏ trùng: ['(())', '(())', '(())', '(())', '(())', '(())', '(())', '(())', '(())', '(())']
           có bỏ trùng  : ['(())']
'((a)'     không bỏ trùng: ['(a)', '(a)']
           có bỏ trùng  : ['(a)']
DFS (bản bỏ trùng) khớp cả 3 ví dụ của đề, không sinh trùng


### Trace DFS cho `"()())"`

Chọn `"()())"` vì nó đủ nhỏ để in hết cây, nhưng có đủ các tình huống: nhánh xóa, nhánh giữ, cắt tỉa P2 và quy tắc bỏ trùng. `count_removals` cho `left_rem = 0, right_rem = 1` (`')'` ở index 4 không ai ghép). Cây quyết định:

```text
i=0 '(' : chỉ GIỮ (del_left = 0 nên không được xóa '(')
├── i=1 ')' XÓA   -> ( ( ) )   -> "(())"  ✔
└── i=1 ')' GIỮ   -> path "()", open_count = 0
    └── i=2 '(' GIỮ -> path "()("
        ├── i=3 ')' XÓA -> i=4 ')' GIỮ -> "()()"  ✔
        └── i=3 ')' GIỮ -> path "()()", open_count = 0
            └── i=4 ')' : XÓA bị cấm (')' ngay trước giống hệt và đã GIỮ)
                          GIỮ bị cấm (open_count = 0 -> P2)   => nhánh chết
```

Nhánh cuối là chỗ đáng chú ý: nếu không có quy tắc bỏ trùng, nhánh "giữ index 3, xóa index 4" sẽ sinh lại `"()()"` lần thứ hai. Kết quả: `["(())", "()()"]`. Output của `trace_dfs` dưới phải khớp với cây này.

In [17]:
def trace_dfs(s: str) -> None:
    left_rem, right_rem = count_removals(s)
    print(f"s={s!r}: left_rem={left_rem}, right_rem={right_rem}")
    n = len(s)
    result, path = [], []

    def dfs(i, open_count, del_left, del_right, prev_kept):
        ind = "  " * i
        if i == n:
            t = "".join(path)
            if del_left == 0 and del_right == 0 and open_count == 0:
                print(f"{ind}KẾT QUẢ {t!r}")
                result.append(t)
            return
        c = s[i]
        can_delete = not (i > 0 and s[i - 1] == c and prev_kept)
        if c == "(":
            if del_left > 0 and can_delete:
                print(f"{ind}i={i} '(' XÓA")
                dfs(i + 1, open_count, del_left - 1, del_right, False)
            print(f"{ind}i={i} '(' GIỮ (open {open_count}->{open_count + 1})")
            path.append(c); dfs(i + 1, open_count + 1, del_left, del_right, True); path.pop()
        elif c == ")":
            if del_right > 0 and can_delete:
                print(f"{ind}i={i} ')' XÓA (del_right {del_right}->{del_right - 1})")
                dfs(i + 1, open_count, del_left, del_right - 1, False)
            elif del_right > 0:
                print(f"{ind}i={i} ')' cấm XÓA: ')' ngay trước đã GIỮ (bỏ trùng)")
            if open_count > 0:
                print(f"{ind}i={i} ')' GIỮ (open {open_count}->{open_count - 1})")
                path.append(c); dfs(i + 1, open_count - 1, del_left, del_right, True); path.pop()
            else:
                print(f"{ind}i={i} ')' cấm GIỮ: open_count = 0 (cắt tỉa P2)")
        else:
            path.append(c); dfs(i + 1, open_count, del_left, del_right, True); path.pop()

    dfs(0, 0, left_rem, right_rem, False)
    print("Đáp án:", result)

trace_dfs("()())")

s='()())': left_rem=0, right_rem=1
i=0 '(' GIỮ (open 0->1)
  i=1 ')' XÓA (del_right 1->0)
    i=2 '(' GIỮ (open 1->2)
      i=3 ')' GIỮ (open 2->1)
        i=4 ')' GIỮ (open 1->0)
          KẾT QUẢ '(())'
  i=1 ')' GIỮ (open 1->0)
    i=2 '(' GIỮ (open 0->1)
      i=3 ')' XÓA (del_right 1->0)
        i=4 ')' GIỮ (open 1->0)
          KẾT QUẢ '()()'
      i=3 ')' GIỮ (open 1->0)
        i=4 ')' cấm XÓA: ')' ngay trước đã GIỮ (bỏ trùng)
        i=4 ')' cấm GIỮ: open_count = 0 (cắt tỉa P2)
Đáp án: ['(())', '()()']


## 12. Cách 4 — DFS hai chiều: quét trái→phải xóa `')'`, đảo chuỗi để xóa `'('`

Đây là lời giải sáng tạo rất nổi tiếng trong phần thảo luận của LeetCode. Thay vì quyết định giữ/xóa **từng ký tự**, nó chỉ ra tay **đúng lúc chuỗi vừa hỏng**.

**Quan sát then chốt.** Quét trái → phải với biến đếm `count` (`'('` cộng 1, `')'` trừ 1). Giả sử lần đầu tiên `count` xuống `-1` tại index `i`. Khi đó:

- Đoạn `t[0..i]` có số `')'` nhiều hơn `'('` **đúng 1**. Mọi chuỗi kết quả hợp lệ đều **bắt buộc** phải xóa ít nhất một `')'` **trong đoạn này** (xóa ký tự phía sau `i` không cứu được đoạn đầu).
- Vậy ta **thử xóa từng `')'`** trong đoạn, mỗi lựa chọn gọi đệ quy cho chuỗi mới. Các ký tự **trước `i` đã được kiểm tra rồi**, nên lời gọi đệ quy **quét tiếp từ `i`** chứ không quét lại từ đầu.
- Nếu quét hết mà `count` không bao giờ âm, chuỗi đã hết `')'` thừa — nhưng có thể còn `'('` thừa. Mẹo: **đảo ngược chuỗi** và **đổi vai** hai loại ngoặc (`')'` thành "mở", `'('` thành "đóng"), rồi chạy **đúng đoạn code cũ** để xóa `'('` thừa. Xong lượt ngược thì đảo lại và ghi nhận đáp án.

Ba câu hỏi cần trả lời trước khi viết code:

1. Đảo chuỗi + đổi vai trò ngoặc hoạt động thế nào? → ngay dưới đây.
2. `last_i` và `last_j` là gì, vì sao cần? → sau cell thí nghiệm.
3. Làm sao không sinh trùng mà không cần `set`? → cùng chỗ với `last_j`.

### Kiến thức nền 6 — Đảo chuỗi `t[::-1]` và tham số `pair`

`t[::-1]` là slicing với bước `-1`: đi từ cuối về đầu → chuỗi đảo ngược. Ví dụ `"(()"[::-1] == ")(("`.

Ta viết **một** hàm quét dùng tham số `pair = (mở, đóng)`:

- Lượt xuôi: `pair = ("(", ")")` — gặp `pair[0]` cộng 1, gặp `pair[1]` trừ 1.
- Lượt ngược: `pair = (")", "(")` — vai trò đổi chỗ.

Với `"(()"` (thừa một `'('` ở **đầu**): quét xuôi, `count` đi `1 → 2 → 1`, **không bao giờ âm** → lượt xuôi không phát hiện được gì. Đảo thành `")(("` và quét với `pair = (")", "(")`: `count` đi `1 → 0 → -1` → âm tại index 2, tức là phát hiện `'('` thừa **y hệt** cách lượt xuôi phát hiện `')'` thừa.

In [18]:
def first_negative(t: str, pair) -> int:
    """Trả về index đầu tiên làm count âm, hoặc -1 nếu không có."""
    count = 0
    for i, c in enumerate(t):
        if c == pair[0]:
            count += 1
        elif c == pair[1]:
            count -= 1
        if count < 0:
            return i
    return -1

print('"(()"[::-1] =', repr("(()"[::-1]))
for t in ["(()", "())", "()())()"]:
    rev = t[::-1]
    print(f"{t!r:10} xuôi: âm tại {first_negative(t, ('(', ')')):2} | "
          f"đảo {rev!r:10} ngược: âm tại {first_negative(rev, (')', '(')):2}")

"(()"[::-1] = ')(('
'(()'      xuôi: âm tại -1 | đảo ')(('      ngược: âm tại  2
'())'      xuôi: âm tại  2 | đảo '))('      ngược: âm tại -1
'()())()'  xuôi: âm tại  4 | đảo ')())()('  ngược: âm tại -1


### Hai biến nhớ `last_i`, `last_j` và quy tắc bỏ trùng

Hàm đệ quy là `remove(t, last_i, last_j, pair)`:

- **`last_i` — quét tiếp từ đâu?** Sau khi xóa một `')'` ở vị trí `j ≤ i`, đoạn đầu của chuỗi mới (tương ứng `t[0..i]` cũ) có `count` đúng bằng 0 và mọi đoạn đầu ngắn hơn đều không âm. Không cần kiểm tra lại → chuỗi mới **quét tiếp từ index `i`** với `count = 0`. Đó là `last_i`.
- **`last_j` — xóa ở đâu?** Nếu lần trước đã xóa ở vị trí `j1`, lần này chỉ được xóa ở vị trí **`≥ j1`**. Lý do: tập vị trí `{j1, j2}` mà xóa theo thứ tự "`j1` trước, `j2` sau" hay "`j2` trước, `j1` sau" đều ra cùng chuỗi; ép các lần xóa có vị trí **tăng dần** thì mỗi tập chỉ được sinh **đúng một lần**. Đó là `last_j`.
- **Bỏ trùng trong một dãy `')'` liền nhau:** chỉ xóa ký tự **đầu dãy** — điều kiện `t[j] == pair[1] and (j == last_j or t[j - 1] != pair[1])`. Vì mỗi lần xóa ta luôn chọn ký tự đầu dãy, ký tự đứng ngay trước `last_j` không bao giờ là `pair[1]`, nên vế `j == last_j` chỉ thực sự "cứu" một trường hợp: **`j = 0`**. Lúc đó `t[j - 1]` là `t[-1]` — Python hiểu index âm là **đếm từ cuối chuỗi** — và nếu ký tự cuối tình cờ là `pair[1]` ta sẽ bỏ sót, ví dụ `")("` sẽ ra `[]` thay vì `[""]` (kiểm chứng ngay bên dưới, ở trace thứ hai).
- **`return` ngay sau vòng `for j`:** chuỗi `t` hiện tại đang hỏng tại `i`; mọi hướng sửa đã được giao cho các lời gọi đệ quy, nên lời gọi hiện tại **không được** quét tiếp hay ghi nhận gì nữa.

In [19]:
def remove_invalid_two_pass(s: str) -> List[str]:
    result = []

    def remove(t, last_i, last_j, pair):
        count = 0
        for i in range(last_i, len(t)):
            if t[i] == pair[0]:
                count += 1
            elif t[i] == pair[1]:
                count -= 1
            if count >= 0:
                continue
            # count < 0: t[0..i] thừa đúng một pair[1] -> thử xóa từng cái (đầu mỗi dãy)
            for j in range(last_j, i + 1):
                if t[j] == pair[1] and (j == last_j or t[j - 1] != pair[1]):
                    remove(t[:j] + t[j + 1:], i, j, pair)
            return                                   # t hiện tại đã hỏng, dừng
        reversed_t = t[::-1]
        if pair[0] == "(":                           # xong lượt xuôi -> làm lượt ngược
            remove(reversed_t, 0, 0, (")", "("))
        else:                                        # xong cả hai lượt -> đảo lại, ghi nhận
            result.append(reversed_t)

    remove(s, 0, 0, ("(", ")"))
    return result

for s, expected in examples:
    got = remove_invalid_two_pass(s)
    print(f"{s!r:11} -> {got}")
    assert sorted(got) == sorted(expected) and len(got) == len(set(got))
print("DFS hai chiều khớp cả 3 ví dụ của đề, không sinh trùng")

'()())()'   -> ['(())()', '()()()']
'(a)())()'  -> ['(a())()', '(a)()()']
')('        -> ['']
DFS hai chiều khớp cả 3 ví dụ của đề, không sinh trùng


### Trace cách 4 cho `"()())()"`

Lượt xuôi, `last_i = 0`:

| i | `t[i]` | count | Hành động |
|:-:|:-:|:-:|---|
| 0 | `(` | 1 | |
| 1 | `)` | 0 | |
| 2 | `(` | 1 | |
| 3 | `)` | 0 | |
| 4 | `)` | **-1** | âm → thử xóa `')'` trong `t[0..4]` |

Các ứng viên `j` (từ `last_j = 0` tới `4`) là những vị trí có `')'`:

- `j = 1`: `t[0] = '('` → đầu dãy → xóa, được `"(())()"`, gọi `remove(..., last_i=4, last_j=1)`.
- `j = 3`: `t[2] = '('` → đầu dãy → xóa, được `"()()()"`, gọi `remove(..., last_i=4, last_j=3)`.
- `j = 4`: `t[3] = ')'` → **cùng dãy** với `j = 3` → **bỏ qua** (xóa nó ra lại `"()()()"`).

Mỗi lời gọi con quét tiếp từ index 4, không còn âm → đảo chuỗi, làm lượt ngược → cũng không âm → đảo lại và ghi nhận. Đáp án `["(())()", "()()()"]` — không cần `set`.

In [20]:
def trace_two_pass(s: str) -> None:
    result = []

    def remove(t, last_i, last_j, pair, depth):
        ind = "  " * depth
        side = "xuôi" if pair[0] == "(" else "ngược"
        print(f"{ind}remove({t!r}, last_i={last_i}, last_j={last_j}, lượt {side})")
        count = 0
        for i in range(last_i, len(t)):
            if t[i] == pair[0]:
                count += 1
            elif t[i] == pair[1]:
                count -= 1
            if count >= 0:
                continue
            print(f"{ind}  i={i}: count=-1 -> phải xóa 1 {pair[1]!r} trong t[{last_j}..{i}]")
            for j in range(last_j, i + 1):
                if t[j] != pair[1]:
                    continue
                if j == last_j or t[j - 1] != pair[1]:
                    print(f"{ind}  xóa j={j}")
                    remove(t[:j] + t[j + 1:], i, j, pair, depth + 2)
                else:
                    print(f"{ind}  bỏ qua j={j} (t[{j - 1}] cũng là {pair[1]!r})")
            return
        reversed_t = t[::-1]
        if pair[0] == "(":
            print(f"{ind}  quét xong không âm -> đảo thành {reversed_t!r}, xử lý '(' thừa")
            remove(reversed_t, 0, 0, (")", "("), depth + 1)
        else:
            print(f"{ind}  sạch cả hai chiều -> đảo lại, ghi nhận {reversed_t!r}")
            result.append(reversed_t)

    remove(s, 0, 0, ("(", ")"), 0)
    print("Đáp án:", result)

trace_two_pass("()())()")

remove('()())()', last_i=0, last_j=0, lượt xuôi)
  i=4: count=-1 -> phải xóa 1 ')' trong t[0..4]
  xóa j=1
    remove('(())()', last_i=4, last_j=1, lượt xuôi)
      quét xong không âm -> đảo thành ')())((', xử lý '(' thừa
      remove(')())((', last_i=0, last_j=0, lượt ngược)
        sạch cả hai chiều -> đảo lại, ghi nhận '(())()'
  xóa j=3
    remove('()()()', last_i=4, last_j=3, lượt xuôi)
      quét xong không âm -> đảo thành ')()()(', xử lý '(' thừa
      remove(')()()(', last_i=0, last_j=0, lượt ngược)
        sạch cả hai chiều -> đảo lại, ghi nhận '()()()'
  bỏ qua j=4 (t[3] cũng là ')')
Đáp án: ['(())()', '()()()']


### Trace thứ hai: `"()))"` — phải xóa **hai** `')'` trong **cùng một dãy**

`"()))"` có `right_rem = 2`: dãy `')))'` ở index 1–3 phải mất 2 ký tự. Theo quy tắc "chỉ xóa phần đầu dãy", cách 4 xóa ký tự đầu dãy (index 1), rồi trong chuỗi mới `"())"` lại xóa ký tự đầu dãy — lúc này đứng ở index 1 = `last_j`. Không có nhánh nào quay về trước `last_j`, và các `')'` đứng sau trong dãy đều bị bỏ qua → chỉ đúng **một** đáp án `"()"`, không trùng.

Cuối cell ta kiểm chứng vai trò của vế `j == last_j`: bản "quên" vế này (chỉ kiểm tra `t[j - 1] != pair[1]`) bị bẫy index âm `t[-1]` ở `j = 0` và trả về `[]` cho `")("`, `")()"`.

In [21]:
trace_two_pass("()))")


def two_pass_missing_guard(s: str) -> List[str]:
    """Bản SAI: điều kiện bỏ trùng thiếu vế j == last_j."""
    result = []

    def remove(t, last_i, last_j, pair):
        count = 0
        for i in range(last_i, len(t)):
            count += 1 if t[i] == pair[0] else (-1 if t[i] == pair[1] else 0)
            if count >= 0:
                continue
            for j in range(last_j, i + 1):
                if t[j] == pair[1] and t[j - 1] != pair[1]:    # thiếu: j == last_j or
                    remove(t[:j] + t[j + 1:], i, j, pair)
            return
        if pair[0] == "(":
            remove(t[::-1], 0, 0, (")", "("))
        else:
            result.append(t[::-1])

    remove(s, 0, 0, ("(", ")"))
    return result

print()
for s in [")(", ")()"]:
    print(f"{s!r:6} bản thiếu vế: {two_pass_missing_guard(s)} | bản đúng: {remove_invalid_two_pass(s)}")

remove('()))', last_i=0, last_j=0, lượt xuôi)
  i=2: count=-1 -> phải xóa 1 ')' trong t[0..2]
  xóa j=1
    remove('())', last_i=2, last_j=1, lượt xuôi)
      i=2: count=-1 -> phải xóa 1 ')' trong t[1..2]
      xóa j=1
        remove('()', last_i=2, last_j=1, lượt xuôi)
          quét xong không âm -> đảo thành ')(', xử lý '(' thừa
          remove(')(', last_i=0, last_j=0, lượt ngược)
            sạch cả hai chiều -> đảo lại, ghi nhận '()'
      bỏ qua j=2 (t[1] cũng là ')')
  bỏ qua j=2 (t[1] cũng là ')')
Đáp án: ['()']

')('   bản thiếu vế: [] | bản đúng: ['']
')()'  bản thiếu vế: [] | bản đúng: ['()']


### Trace thứ ba: `"()())(()"` — cần **cả hai lượt**

`count_removals("()())(()")` cho `left_rem = 1, right_rem = 1`: một `')'` thừa ở index 4 và một `'('` thừa trong phần `"(()"` cuối.

- **Lượt xuôi** giống hệt ví dụ trước: âm tại `i = 4`, xóa `j = 1` hoặc `j = 3`, bỏ qua `j = 4`.
- Mỗi chuỗi con (`"(())(()"`, `"()()(()"`) quét xuôi sạch → **đảo chuỗi**. Ví dụ `"(())(()"` đảo thành `")(())(("`. Quét với `pair = (")", "(")`: `count` đi `1 → 0 → -1` → âm tại `i = 2` → xóa `'('` ở `j = 1`, còn `j = 2` cùng dãy `'(('` nên bỏ qua.
- Chuỗi sau khi xóa, quét ngược sạch → **đảo lại** → ghi nhận `"(())()"`. Nhánh còn lại tương tự cho `"()()()"`.

Chú ý: `'('` bị xóa ở lượt ngược (index 1 của chuỗi đảo) chính là dấu `'('` **thứ hai** của cụm `"(("` trong phần `"(()"` cuối chuỗi gốc (index 6 của `s`). Vì chuỗi bị đảo, "đầu dãy" ở lượt ngược là ký tự **cuối** dãy ở chuỗi gốc — không sao cả: mỗi dãy vẫn chỉ có đúng **một** đại diện được xóa.

In [22]:
trace_two_pass("()())(()")

remove('()())(()', last_i=0, last_j=0, lượt xuôi)
  i=4: count=-1 -> phải xóa 1 ')' trong t[0..4]
  xóa j=1
    remove('(())(()', last_i=4, last_j=1, lượt xuôi)
      quét xong không âm -> đảo thành ')(())((', xử lý '(' thừa
      remove(')(())((', last_i=0, last_j=0, lượt ngược)
        i=2: count=-1 -> phải xóa 1 '(' trong t[0..2]
        xóa j=1
          remove(')())((', last_i=2, last_j=1, lượt ngược)
            sạch cả hai chiều -> đảo lại, ghi nhận '(())()'
        bỏ qua j=2 (t[1] cũng là '(')
  xóa j=3
    remove('()()(()', last_i=4, last_j=3, lượt xuôi)
      quét xong không âm -> đảo thành ')(()()(', xử lý '(' thừa
      remove(')(()()(', last_i=0, last_j=0, lượt ngược)
        i=2: count=-1 -> phải xóa 1 '(' trong t[0..2]
        xóa j=1
          remove(')()()(', last_i=2, last_j=1, lượt ngược)
            sạch cả hai chiều -> đảo lại, ghi nhận '()()()'
        bỏ qua j=2 (t[1] cũng là '(')
  bỏ qua j=4 (t[3] cũng là ')')
Đáp án: ['(())()', '()()()']


## 13. Code nộp LeetCode

Chọn **Cách 3** (đếm trước + DFS cắt tỉa + bỏ trùng) làm bản nộp chính: dễ giải thích trong phỏng vấn, mỗi dòng code tương ứng một ý đã học. Lưu ý khi nộp:

- LeetCode đã `from typing import List` sẵn; ở notebook ta tự import ở đầu (mục 2).
- Hàm đệ quy lồng bên trong method, dùng chung `s`, `n`, `path`, `result` của method.
- Trả về `list`.

Bên dưới còn có bản **Cách 4** (đặt tên `SolutionTwoPass` để không đè lên `Solution`; khi nộp thì đổi tên lớp thành `Solution`). Cả hai được test bằng ví dụ của đề và các edge case: chỉ có chữ cái, đã hợp lệ sẵn, toàn `'('`, toàn `')'`, chữ cái xen giữa.

In [23]:
class Solution:
    def removeInvalidParentheses(self, s: str) -> List[str]:
        # 1) Đếm số '(' và ')' bắt buộc phải xóa
        open_count = right_rem = 0
        for c in s:
            if c == "(":
                open_count += 1
            elif c == ")":
                if open_count > 0:
                    open_count -= 1
                else:
                    right_rem += 1
        left_rem = open_count

        # 2) DFS giữ/xóa từng ký tự, có hạn mức + cắt tỉa + bỏ trùng
        n = len(s)
        result, path = [], []

        def dfs(i, opened, del_left, del_right, prev_kept):
            if i == n:
                if del_left == 0 and del_right == 0 and opened == 0:
                    result.append("".join(path))
                return
            c = s[i]
            can_delete = not (i > 0 and s[i - 1] == c and prev_kept)
            if c == "(":
                if del_left > 0 and can_delete:
                    dfs(i + 1, opened, del_left - 1, del_right, False)
                path.append(c)
                dfs(i + 1, opened + 1, del_left, del_right, True)
                path.pop()
            elif c == ")":
                if del_right > 0 and can_delete:
                    dfs(i + 1, opened, del_left, del_right - 1, False)
                if opened > 0:
                    path.append(c)
                    dfs(i + 1, opened - 1, del_left, del_right, True)
                    path.pop()
            else:
                path.append(c)
                dfs(i + 1, opened, del_left, del_right, True)
                path.pop()

        dfs(0, 0, left_rem, right_rem, False)
        return result


tests = examples + [
    ("abc", ["abc"]),               # chỉ có chữ cái
    ("(a)(b)", ["(a)(b)"]),         # đã hợp lệ sẵn
    ("((((", [""]),                 # toàn '('
    ("))))", [""]),                 # toàn ')'
    ("x(", ["x"]),
    ("(((k()((", ["k()", "(k)"]),
    (")a)b(c)", ["ab(c)"]),
    ("n", ["n"]),
]
sol = Solution()
for s, expected in tests:
    got = sol.removeInvalidParentheses(s)
    assert sorted(got) == sorted(expected), (s, got, expected)
    assert len(got) == len(set(got)), ("trùng", s, got)
print("class Solution PASS", len(tests), "test")

class Solution PASS 11 test


In [24]:
class SolutionTwoPass:           # khi nộp: đổi tên thành Solution
    def removeInvalidParentheses(self, s: str) -> List[str]:
        result = []

        def remove(t, last_i, last_j, pair):
            count = 0
            for i in range(last_i, len(t)):
                if t[i] == pair[0]:
                    count += 1
                elif t[i] == pair[1]:
                    count -= 1
                if count >= 0:
                    continue
                for j in range(last_j, i + 1):
                    if t[j] == pair[1] and (j == last_j or t[j - 1] != pair[1]):
                        remove(t[:j] + t[j + 1:], i, j, pair)
                return
            reversed_t = t[::-1]
            if pair[0] == "(":
                remove(reversed_t, 0, 0, (")", "("))
            else:
                result.append(reversed_t)

        remove(s, 0, 0, ("(", ")"))
        return result


sol2 = SolutionTwoPass()
for s, expected in tests:
    got = sol2.removeInvalidParentheses(s)
    assert sorted(got) == sorted(expected), (s, got, expected)
    assert len(got) == len(set(got)), ("trùng", s, got)
print("class SolutionTwoPass PASS", len(tests), "test")

class SolutionTwoPass PASS 11 test


## 14. Kiểm thử chéo với brute force trên chuỗi ngẫu nhiên

Ví dụ của đề quá ít để tin tưởng. Ta sinh **2000 chuỗi ngẫu nhiên** dài 0–12 ký tự (đủ ngắn để brute force `O(2^n · n)` chạy nhanh), trộn ngoặc với **chữ cái**, rồi yêu cầu:

1. Cả 5 lời giải (BFS, DFS bản `set`, DFS bản bỏ trùng, DFS hai chiều, `class Solution`) cho **cùng tập đáp án** với brute force (so bằng `sorted`).
2. Các bản **không dùng `set`** (DFS bỏ trùng, hai chiều, `Solution`) **không bao giờ** sinh chuỗi trùng.

`random.seed(301)` cố định "hạt giống" ngẫu nhiên để lần chạy nào cũng ra cùng bộ test (dễ tái hiện khi có lỗi). Chuỗi rỗng không nằm trong constraints (`1 <= s.length`) nhưng thử thêm cũng vô hại.

In [25]:
random.seed(301)
solvers = {
    "bfs": remove_invalid_bfs,
    "dfs_set": remove_invalid_dfs_set,
    "dfs_skip": remove_invalid_dfs,
    "two_pass": remove_invalid_two_pass,
    "Solution": Solution().removeInvalidParentheses,
}
no_set = {"dfs_skip", "two_pass", "Solution"}

for _ in range(2000):
    length = random.randint(0, 12)
    s = "".join(random.choice("(()))ab") for _ in range(length))
    expected = sorted(remove_invalid_bruteforce(s))
    for name, solve in solvers.items():
        got = solve(s)
        assert sorted(got) == expected, (name, s, got, expected)
        if name in no_set:
            assert len(got) == len(set(got)), (name, "sinh trùng", s, got)
print("2000 chuỗi ngẫu nhiên: cả 5 lời giải khớp brute force, các bản không dùng set không sinh trùng")

2000 chuỗi ngẫu nhiên: cả 5 lời giải khớp brute force, các bản không dùng set không sinh trùng


## 15. Vì sao đúng?

**Cách 1 — Bitmask.** Duyệt **đủ** `2^n` tập ký tự giữ lại nên không sót chuỗi nào; chỉ giữ chuỗi hợp lệ **dài nhất** ⇔ xóa ít nhất; dùng `set` nên không trùng.

**Cách 2 — BFS theo tầng.** *Bất biến:* khi bắt đầu xét tầng `k`, `level` chứa **đúng tất cả** các chuỗi **khác nhau** có được bằng cách xóa `k` ngoặc. Tầng `0..k-1` không có chuỗi hợp lệ → cần ít nhất `k` lần xóa; tầng `k` có → `k` là tối thiểu; lấy trọn tầng `k` → đủ mọi đáp án.

**Cách 3 — Đếm + DFS.**
1. `left_rem + right_rem` là số lần xóa tối thiểu (chứng minh ở mục 9), và mọi đáp án xóa đúng `left_rem` dấu `'('`, `right_rem` dấu `')'`.
2. DFS (không có quy tắc bỏ trùng) liệt kê **mọi** cách chọn đúng chừng ấy ngoặc để xóa, chỉ trừ các nhánh bị P2 cắt — mà nhánh bị cắt chắc chắn hỏng → không sót đáp án.
3. *Bất biến:* `open_count` = `balance` của `path` và luôn `≥ 0` (nhờ P2). Ở lá, `open_count == 0` → `path` hợp lệ.

**Bổ đề "ranh giới" — lý do quy tắc bỏ trùng là đủ (cho cả cách 3 và cách 4).** Giá trị nhỏ nhất của `balance` trên các đoạn đầu của `s` đúng bằng `-right_rem` (mỗi lần `right_rem` tăng là một lần `balance` xuống một "đáy" mới). Gọi `m` là **vị trí cuối cùng** mà `balance` chạm đáy `-right_rem` (nếu `right_rem = 0` thì đáy là 0, và `m = -1` khi `balance` chưa từng trở về 0 sau khi bắt đầu). Khi đó, trong **mọi** cách xóa tối thiểu:

- **mọi `')'` bị xóa nằm ở vị trí `≤ m`**, và **mọi `'('` bị xóa nằm sau `m`**.

*Lý do:* đoạn đầu `s[0..m]` có `balance = -right_rem`. Sau khi xóa, đoạn này phải có `balance ≥ 0`, tức là `-right_rem + (số ')' xóa trong đoạn) - (số '(' xóa trong đoạn) ≥ 0`. Mà tổng số `')'` được xóa chỉ là `right_rem` → **tất cả** chúng phải nằm trong đoạn đầu, và **không** `'('` nào được xóa ở đó.

*Hệ quả:* trong đoạn đầu chỉ có `')'` bị xóa, còn `'('` và chữ cái đứng yên như **cọc mốc**. Chuỗi kết quả của đoạn đầu vì thế chỉ phụ thuộc vào **số** `')'` bị xóa trong **mỗi dãy `')'` liền nhau** — "chỉ xóa phần đầu mỗi dãy" là cách biểu diễn **duy nhất**. Đoạn sau tương tự với `'('`. Vậy hai cách xóa khác nhau (đã chuẩn hóa) luôn cho **hai chuỗi khác nhau** → không cần `set`.

**Cách 4 — DFS hai chiều.** Lần đầu `count` âm tại `i` buộc phải xóa một `')'` trong `t[0..i]`; ta thử **mọi** lựa chọn khác nhau (mỗi dãy một đại diện) → không sót. Số lần `count` âm đúng bằng `right_rem` → số `')'` bị xóa là tối thiểu; lượt ngược tương tự cho `'('`. Không trùng nhờ: `last_j` ép vị trí xóa tăng dần + chỉ xóa đầu dãy + bổ đề ranh giới (lượt xuôi chỉ đụng `')'`, lượt ngược chỉ đụng `'('`).

Cell dưới kiểm chứng bổ đề: liệt kê **mọi** tập vị trí bị xóa của các cách xóa tối thiểu (dùng lại ý bitmask), in cho `"()())(()"`, rồi kiểm tra trên 1500 chuỗi ngẫu nhiên.

In [26]:
def last_min_position(s: str) -> int:
    balance, low, m = 0, 0, -1
    for i, c in enumerate(s):
        if c == "(":
            balance += 1
        elif c == ")":
            balance -= 1
        if balance <= low:            # chạm (hoặc xuống dưới) đáy -> cập nhật m
            low, m = balance, i
    return m


def optimal_deletion_sets(s: str) -> List[List[int]]:
    n = len(s)
    best_len, sets = -1, []
    for mask in range(1 << n):
        t = "".join(s[i] for i in range(n) if (mask >> i) & 1)
        if is_valid(t):
            if len(t) > best_len:
                best_len, sets = len(t), []
            if len(t) == best_len:
                sets.append([i for i in range(n) if not (mask >> i) & 1])
    return sets


s = "()())(()"
m = last_min_position(s)
print(f"s = {s!r}, m = {m}")
for d in optimal_deletion_sets(s):
    print("  xóa index", d, "->", [s[i] for i in d], "-> còn", repr("".join(c for i, c in enumerate(s) if i not in d)))

random.seed(15)
for _ in range(1500):
    s = "".join(random.choice("(()a)") for _ in range(random.randint(0, 10)))
    m = last_min_position(s)
    for d in optimal_deletion_sets(s):
        assert all(i <= m for i in d if s[i] == ")"), s
        assert all(i > m for i in d if s[i] == "("), s
print("Bổ đề ranh giới đúng trên 1500 chuỗi ngẫu nhiên")

s = '()())(()', m = 4
  xóa index [4, 6] -> [')', '('] -> còn '()()()'
  xóa index [3, 6] -> [')', '('] -> còn '()()()'
  xóa index [1, 6] -> [')', '('] -> còn '(())()'
  xóa index [4, 5] -> [')', '('] -> còn '()()()'
  xóa index [3, 5] -> [')', '('] -> còn '()()()'
  xóa index [1, 5] -> [')', '('] -> còn '(())()'


Bổ đề ranh giới đúng trên 1500 chuỗi ngẫu nhiên


## 16. Độ phức tạp và so sánh các cách

Nhắc lại: `n = len(s) ≤ 25`, `p` = số ngoặc `≤ 20`, `k = left_rem + right_rem` = số lần xóa tối thiểu.

| Cách | Ý tưởng | Time (xấu nhất) | Space | Nhận xét |
|---|---|---|---|---|
| 1. Bitmask | thử mọi tập ký tự giữ lại | `O(2^n · n)` | `O(n)` + đáp án | chắc chắn đúng; chỉ làm trọng tài |
| 2. BFS theo tầng | xóa 0, 1, 2... ký tự, dừng ở tầng đầu có đáp án | `O(p · n · Σ_{j≤k} C(p, j))` ≤ `O(p · n · 2^p)` | `O(n ·` số chuỗi đã thăm`)` | dễ nghĩ, tổng quát (Bài 6); tốn bộ nhớ |
| 3. Đếm + DFS | biết trước xóa bao nhiêu mỗi loại, giữ/xóa từng ký tự, cắt tỉa | `O(2^p · n)` | `O(n)` đệ quy + đáp án | **cách chính**; nhanh, dễ giải thích |
| 4. DFS hai chiều | chỉ ra tay khi `count` âm; đảo chuỗi cho `'('` | `O(2^p · n)` | `O(n · k)` (mỗi tầng đệ quy giữ một bản sao chuỗi, độ sâu cỡ `k`) + đáp án | thường nhanh nhất; khó nghĩ ra |

Cả 4 cách đều **hàm mũ** ở trường hợp xấu nhất — điều khó tránh vì **số đáp án** tự nó có thể rất lớn — nhưng khác nhau một trời một vực về lượng việc vô ích. Cell dưới đếm "lượng việc" (số chuỗi BFS sinh ra, số lời gọi đệ quy) trên vài input; dòng cuối là chuỗi `'(a)())()(()((b)(()(c)(('` dài 23 ký tự với **đúng 20 ngoặc** — sát giới hạn đề. Cột "DFS không quota" là số nút của cây giữ/xóa đầy đủ (mỗi ngoặc 2 nhánh, không hạn mức, không cắt tỉa), tính bằng công thức thay vì chạy thật.

Đọc dòng cuối của bảng kết quả: để ra **18** đáp án, brute force phải thử **8 388 608** mask, cây giữ/xóa đầy đủ có **2 232 321** nút, BFS sinh **10 522** chuỗi, DFS bản `set` cần **2 060** lời gọi, DFS bỏ trùng còn **528**, và DFS hai chiều chỉ **39** lời gọi. Hạn mức (P1) và cắt tỉa (P2) cắt đi hơn 99,9% cây; quy tắc bỏ trùng cắt tiếp khoảng 3/4 số lời gọi còn lại.

In [27]:
def work_bfs(s: str) -> int:
    level, visited, total = [s], {s}, 1
    while not any(is_valid(t) for t in level):
        next_level = []
        for t in level:
            for i, c in enumerate(t):
                if c in "()":
                    child = t[:i] + t[i + 1:]
                    if child not in visited:
                        visited.add(child)
                        next_level.append(child)
        level = next_level
        total += len(level)
    return total


def work_full_tree(s: str) -> int:
    # số nút của cây giữ/xóa đầy đủ: tầng i có 2^(số ngoặc trong s[:i]) nút
    return sum(2 ** sum(c in "()" for c in s[:i]) for i in range(len(s) + 1))


def work_dfs(s: str, skip: bool) -> int:
    del_l, del_r = count_removals(s)
    n, calls = len(s), 0

    def dfs(i, opened, dl, dr, prev_kept):
        nonlocal calls
        calls += 1
        if i == n:
            return
        c = s[i]
        can_delete = not (skip and i > 0 and s[i - 1] == c and prev_kept)
        if c == "(":
            if dl > 0 and can_delete:
                dfs(i + 1, opened, dl - 1, dr, False)
            dfs(i + 1, opened + 1, dl, dr, True)
        elif c == ")":
            if dr > 0 and can_delete:
                dfs(i + 1, opened, dl, dr - 1, False)
            if opened > 0:
                dfs(i + 1, opened - 1, dl, dr, True)
        else:
            dfs(i + 1, opened, dl, dr, True)

    dfs(0, 0, del_l, del_r, False)
    return calls


def work_two_pass(s: str) -> int:
    calls = 0

    def remove(t, last_i, last_j, pair):
        nonlocal calls
        calls += 1
        count = 0
        for i in range(last_i, len(t)):
            count += 1 if t[i] == pair[0] else (-1 if t[i] == pair[1] else 0)
            if count >= 0:
                continue
            for j in range(last_j, i + 1):
                if t[j] == pair[1] and (j == last_j or t[j - 1] != pair[1]):
                    remove(t[:j] + t[j + 1:], i, j, pair)
            return
        if pair[0] == "(":
            remove(t[::-1], 0, 0, (")", "("))

    remove(s, 0, 0, ("(", ")"))
    return calls


print(f"{'s':27}{'n':>3}{'p':>3}{'k':>3}{'2^n mask':>10}{'BFS':>7}{'cây đủ':>9}{'DFS set':>8}{'DFS skip':>9}{'2 chiều':>8}{'#đáp án':>8}")
for s in ["()())()", ")(", "()())(()", ")()(()()))(a)((()(()", "(a)())()(()((b)(()(c)(("]:
    p = sum(c in "()" for c in s)
    k = sum(count_removals(s))
    print(f"{s!r:27}{len(s):>3}{p:>3}{k:>3}{2 ** len(s):>10}{work_bfs(s):>7}{work_full_tree(s):>9}"
          f"{work_dfs(s, False):>8}{work_dfs(s, True):>9}{work_two_pass(s):>8}{len(Solution().removeInvalidParentheses(s)):>8}")

s                            n  p  k  2^n mask    BFS   cây đủ DFS set DFS skip 2 chiều #đáp án
'()())()'                    7  7  1       128      7      255      18       15       5       2
')('                         2  2  2         4      4        7       4        4       4       1
'()())(()'                   8  8  2       256     22      511      44       31       7       2
')()(()()))(a)((()(()'      20 19  5   1048576   2392  1050623     568      203      23       6
'(a)())()(()((b)(()(c)(('   23 20  6   8388608  10522  2232321    2060      528      39      18

## 17. Lỗi thường gặp

### Lỗi 1: Coi chữ cái là `')'` khi kiểm tra hợp lệ

Code quen tay từ các bài chỉ có ngoặc:

```python
balance += 1 if c == "(" else -1      # chữ cái bị trừ như ')'
```

Với `"(a)"` → balance `1 → 0 → -1` → báo **không hợp lệ**, trong khi chuỗi này hợp lệ. Mọi lời giải dùng hàm kiểm tra này đều sai theo, ví dụ BFS sẽ đi tìm những chuỗi mà chữ cái "đóng vai" `')'`.

**Sửa:** dùng `elif c == ")"`, để chữ cái không rơi vào nhánh nào.

In [28]:
def is_valid_wrong(t: str) -> bool:
    balance = 0
    for c in t:
        balance += 1 if c == "(" else -1     # SAI: chữ cái cũng bị trừ
        if balance < 0:
            return False
    return balance == 0


def bfs_with(check, s: str) -> List[str]:
    level, visited = [s], {s}
    while True:
        valid = [t for t in level if check(t)]
        if valid:
            return valid
        next_level = []
        for t in level:
            for i, c in enumerate(t):
                if c in "()" and (t[:i] + t[i + 1:]) not in visited:
                    visited.add(t[:i] + t[i + 1:])
                    next_level.append(t[:i] + t[i + 1:])
        level = next_level

print("is_valid_wrong('(a)') =", is_valid_wrong("(a)"), "| is_valid('(a)') =", is_valid("(a)"))
print("SAI :", sorted(bfs_with(is_valid_wrong, "(a)())()")))
print("ĐÚNG:", sorted(bfs_with(is_valid, "(a)())()")))

is_valid_wrong('(a)') = False | is_valid('(a)') = True
SAI : ['(a()()']
ĐÚNG: ['(a())()', '(a)()()']


### Lỗi 2: BFS dừng sai chỗ

Hai kiểu sai đối xứng nhau:

- **Trả về ngay chuỗi hợp lệ đầu tiên** gặp được → chỉ có **một** đáp án, thiếu các chuỗi khác cùng tầng. Đề đòi **tất cả**.
- **Không dừng** ở tầng đầu tiên có đáp án, cứ gom chuỗi hợp lệ ở mọi tầng → lẫn cả những chuỗi xóa **nhiều hơn** mức tối thiểu (ví dụ `"()()"` với input `"()())()"`).

**Sửa:** kiểm tra **trọn một tầng**, gom tất cả chuỗi hợp lệ của tầng đó, rồi **dừng**.

In [29]:
def bfs_return_first(s: str) -> List[str]:          # SAI kiểu 1
    level, visited = [s], {s}
    while True:
        for t in level:
            if is_valid(t):
                return [t]                           # trả về quá sớm
        next_level = []
        for t in level:
            for i, c in enumerate(t):
                child = t[:i] + t[i + 1:]
                if c in "()" and child not in visited:
                    visited.add(child)
                    next_level.append(child)
        level = next_level


def bfs_never_stop(s: str) -> List[str]:            # SAI kiểu 2
    level, visited, found = [s], {s}, []
    while level:
        found += [t for t in level if is_valid(t)]   # gom ở MỌI tầng
        next_level = []
        for t in level:
            for i, c in enumerate(t):
                child = t[:i] + t[i + 1:]
                if c in "()" and child not in visited:
                    visited.add(child)
                    next_level.append(child)
        level = next_level
    return found

s = "()())()"
print("Trả về sớm :", bfs_return_first(s))
print("Không dừng :", bfs_never_stop(s))
print("ĐÚNG       :", remove_invalid_bfs(s))

Trả về sớm : ['(())()']
Không dừng : ['(())()', '()()()', '()()', '(())', '()', '']
ĐÚNG       : ['(())()', '()()()']


### Lỗi 3: DFS quên cắt tỉa P2 — giữ `')'` cả khi `open_count == 0`

Nếu cho phép giữ `')'` khi chưa có `'('` nào mở, `open_count` có thể xuống âm rồi **quay lại 0** ở cuối chuỗi. Khi đó điều kiện ở lá `open_count == 0` vẫn đúng, và một chuỗi **không hợp lệ** lọt vào đáp án!

Ví dụ `"())("` (`left_rem = 1, right_rem = 1`): nhánh "xóa `'('` index 0, giữ `')'` index 1, xóa `')'` index 2, giữ `'('` index 3" cho `path = ")("` với `open_count` đi `-1 → 0` → được ghi nhận như một đáp án.

**Sửa:** chỉ giữ `')'` khi `open_count > 0`. Nhát cắt này vừa **đúng** (bỏ nhánh hỏng) vừa **nhanh** (bỏ sớm cả cây con).

In [30]:
def dfs_without_p2(s: str) -> List[str]:
    left_rem, right_rem = count_removals(s)
    n, result, path = len(s), set(), []

    def dfs(i, open_count, del_left, del_right):
        if i == n:
            if del_left == 0 and del_right == 0 and open_count == 0:
                result.add("".join(path))
            return
        c = s[i]
        if c in "()":
            if c == "(" and del_left > 0:
                dfs(i + 1, open_count, del_left - 1, del_right)
            if c == ")" and del_right > 0:
                dfs(i + 1, open_count, del_left, del_right - 1)
        path.append(c)                                     # SAI: giữ mọi ký tự, kể cả ')' khi open_count = 0
        dfs(i + 1, open_count + (c == "(") - (c == ")"), del_left, del_right)
        path.pop()

    dfs(0, 0, left_rem, right_rem)
    return sorted(result)

print("Thiếu P2:", dfs_without_p2("())("))
print("ĐÚNG    :", sorted(remove_invalid_dfs("())(")))

Thiếu P2: ['()', ')(']
ĐÚNG    : ['()']


### Các lỗi khác hay gặp

| Lỗi | Hậu quả | Sửa |
|---|---|---|
| Trả `[]` khi phải xóa hết ngoặc | sai ví dụ 3: `")("` phải ra `[""]` | chuỗi rỗng **cũng là** chuỗi hợp lệ; thuật toán đúng sẽ tự sinh ra `""` |
| Không bỏ trùng | `"()())()"` trả `["(())()", "()()()", "()()()"]` | dùng `set`, hoặc quy tắc "chỉ xóa phần đầu dãy" |
| Quên `path.pop()` sau khi đi sâu | `path` dính ký tự của nhánh trước → chuỗi rác | luôn đủ 3 nhịp **chọn → đi sâu → hoàn tác** |
| Cách 4 quên `return` sau vòng `for j` | lời gọi tiếp tục quét chuỗi đang hỏng; `"()())()"` trả về 7 chuỗi, có cả `"()())"` không hợp lệ và chuỗi trùng | `return` ngay sau vòng `for j` |
| Cách 4 quên đảo lại ở lượt ngược | trả về chuỗi bị **ngược** | ở lượt ngược ghi nhận `t[::-1]`, không phải `t` |
| Cách 4 thiếu vế `j == last_j` | bẫy `t[-1]`: `")("` ra `[]` | giữ đủ `j == last_j or t[j - 1] != pair[1]` |
| Xóa cả chữ cái (BFS) | tầng phình to vô ích, chậm hơn nhiều | chỉ sinh con khi `t[i] in "()"` |
| Dùng `list.pop(0)` làm hàng đợi | mỗi lần `O(len)` | `deque.popleft()` hoặc BFS theo tầng bằng list |

## 18. Mẹo nhận dạng pattern

- **"Xóa ít nhất ... trả về TẤT CẢ kết quả"** → nghĩ ngay tới **BFS theo tầng**: mỗi bước = một thao tác, tầng đầu tiên chạm đích = số thao tác ít nhất. Cùng họ với #127 Word Ladder.
- **Ngoặc một loại** → kiểm tra hợp lệ bằng **biến đếm `balance`**; nhiều loại ngoặc (`()[]{}`) → phải dùng **stack** (Bài 6).
- **Đếm được trước "phải làm bao nhiêu"** → biến bài "tìm tối thiểu" thành **liệt kê có hạn mức** (backtracking với quota) — cắt bỏ hẳn các nhánh làm thừa.
- **Constraints rất nhỏ** (`n ≤ 25`, `≤ 20` ngoặc) + "trả về tất cả" → lời giải **hàm mũ** là chấp nhận được; việc của ta là **cắt tỉa** cho thông minh.
- **Phần tử giống nhau liền kề gây trùng** → "chỉ chọn phần tử **đầu** của mỗi dãy trùng". Mẹo này xuất hiện ở #40 Combination Sum II, #47 Permutations II, #90 Subsets II.
- **Xử lý một chiều xong rồi muốn làm y hệt cho chiều kia** → **đảo chuỗi + đổi vai** để tái dùng code (giống hai lượt quét của #32).

## 19. Biến thể và bài liên quan

| Bài | Liên hệ với #301 |
|---|---|
| #20 Valid Parentheses (Easy) | kiểm tra hợp lệ với 3 loại ngoặc → stack |
| #22 Generate Parentheses (Medium) | backtracking sinh chuỗi hợp lệ, cắt tỉa bằng số `'('`/`')'` đã dùng — giống P2 |
| #32 Longest Valid Parentheses (Hard) | biến đếm `balance`, quét hai chiều |
| #921 Minimum Add to Make Parentheses Valid (Medium) | cùng lượt đếm: đáp án = `left_rem + right_rem` (thêm một ngoặc "đối tác" cho mỗi ngoặc thừa) |
| #1249 Minimum Remove to Make Valid Parentheses (Medium) | chỉ cần **một** chuỗi hợp lệ sau khi xóa ít nhất → `O(n)` — **Bài tập 5** |
| #678 Valid Parenthesis String (Medium) | thêm ký tự `'*'` linh hoạt, theo dõi khoảng `[min, max]` của `balance` |
| #127 Word Ladder (Hard) | BFS theo tầng trên chuỗi, `visited` để không thăm lại |
| #40 / #90 | quy tắc bỏ trùng "chỉ lấy phần tử đầu dãy" |

**Biến thể tự nghĩ:** nếu chuỗi có **hai loại** ngoặc `()` và `[]` thì sao? Lượt đếm `left_rem/right_rem` không còn đúng (ví dụ `"([)]"` mỗi loại đều cân bằng mà vẫn không hợp lệ), nhưng **BFS theo tầng** vẫn chạy được nếu thay hàm kiểm tra bằng stack — đó là **Bài tập 6**.

## 20. Bài tập tự luyện

Làm theo thứ tự, tự làm trước rồi mới xem **Gợi ý** (mục 21) và **Đáp án** (mục 22, cuối notebook).

### Bài 1 — Đoán output (không chạy code)

`count_removals` trả về gì với từng chuỗi dưới? Từ đó suy ra **mọi** đáp án của #301 dài bao nhiêu ký tự.

```python
count_removals("())(")
count_removals("(a(b")
count_removals("))((")
count_removals("a)b(c)d")
```

### Bài 2 — Tính tay BFS theo tầng

Với `s = "(()("`, liệt kê **tầng 1** và **tầng 2** của BFS (ghi rõ chuỗi nào bị loại vì đã có trong `visited`), đánh dấu chuỗi hợp lệ, và cho biết đáp án cuối cùng.

### Bài 3 — Điền TODO: DFS có hạn mức (cách 3, bản dùng `set`)

Hoàn thiện hàm dưới theo đúng mục 9 và mục 11. Tự viết lại lượt đếm, đừng gọi `count_removals`.

In [31]:
def exercise_remove_invalid_dfs(s: str) -> List[str]:
    # TODO 1: đếm left_rem, right_rem bằng một lượt quét
    left_rem, right_rem = 0, 0
    n = len(s)
    result = set()
    path = []

    def dfs(i, open_count, del_left, del_right):
        if i == n:
            # TODO 2: nếu đã xóa đủ và open_count == 0 -> thêm "".join(path) vào result
            return
        c = s[i]
        if c == "(":
            # TODO 3: nhánh XÓA (khi del_left > 0) và nhánh GIỮ
            pass
        elif c == ")":
            # TODO 4: nhánh XÓA (khi del_right > 0) và nhánh GIỮ (chỉ khi open_count > 0)
            pass
        else:
            # TODO 5: chữ cái luôn giữ (nhớ hoàn tác path)
            pass

    dfs(0, 0, left_rem, right_rem)
    return list(result)

# Tự làm xong rồi mới mở test:
# assert sorted(exercise_remove_invalid_dfs("()())()")) == ["(())()", "()()()"]
# assert sorted(exercise_remove_invalid_dfs("(a)())()")) == ["(a())()", "(a)()()"]
# assert exercise_remove_invalid_dfs(")(") == [""]

### Bài 4 — Tự viết lại: BFS theo tầng

Không nhìn lại mục 8, tự viết BFS theo tầng có `visited`. Nhớ ba điều: chỉ xóa **ngoặc**, kiểm tra **trọn tầng**, **dừng** ở tầng đầu tiên có đáp án.

In [32]:
def exercise_bfs(s: str) -> List[str]:
    # TODO 1: level = [s], visited = {s}
    # TODO 2: lặp: lọc các chuỗi hợp lệ trong level; có thì trả về ngay
    # TODO 3: sinh tầng sau: xóa từng ký tự NGOẶC, chỉ nhận chuỗi chưa có trong visited
    raise NotImplementedError

# Gợi ý test:
# assert sorted(exercise_bfs("()())()")) == ["(())()", "()()()"]
# assert sorted(exercise_bfs("(a)())()")) == ["(a())()", "(a)()()"]
# assert exercise_bfs(")(") == [""]
# assert sorted(exercise_bfs("(()(")) == ["()"]

### Bài 5 — Biến thể (LeetCode #1249): chỉ cần **một** chuỗi hợp lệ

Xóa **ít nhất** số ngoặc để chuỗi hợp lệ, nhưng chỉ trả về **một** chuỗi kết quả (bất kỳ). Yêu cầu `O(n)`.

```python
min_remove_to_make_valid("lee(t(c)o)de)")  -> "lee(t(c)o)de"    # một đáp án hợp lệ
min_remove_to_make_valid("a)b(c)d")        -> "ab(c)d"
min_remove_to_make_valid("))((")           -> ""
```

Tự kiểm tra: kết quả phải hợp lệ, dài đúng `len(s) - (left_rem + right_rem)`, và nằm trong danh sách mà `Solution().removeInvalidParentheses(s)` trả về.

In [33]:
def min_remove_to_make_valid(s: str) -> str:
    # TODO: tìm các ')' không có ai ghép và các '(' còn mở ở cuối,
    #       rồi dựng chuỗi mới bỏ đúng những vị trí đó
    pass

# Gợi ý test:
# for s in ["lee(t(c)o)de)", "a)b(c)d", "))((", "(a(b(c)d)"]:
#     ans = min_remove_to_make_valid(s)
#     assert is_valid(ans) and len(ans) == len(s) - sum(count_removals(s))
#     assert ans in set(Solution().removeInvalidParentheses(s))

### Bài 6 — Mini challenge: hai loại ngoặc `()` và `[]`

Chuỗi gồm chữ cái và 4 loại ký tự ngoặc `( ) [ ]`. Chuỗi hợp lệ khi mọi ngoặc được đóng **đúng loại** và **đúng thứ tự lồng nhau**. Xóa ít nhất số ngoặc để hợp lệ, trả về **tất cả** chuỗi kết quả khác nhau.

```python
remove_invalid_two_kinds("([)]")  -> ["()", "[]"]      # thứ tự tùy ý
remove_invalid_two_kinds("(a]")   -> ["a"]
remove_invalid_two_kinds("()[]")  -> ["()[]"]
remove_invalid_two_kinds("([)")   -> ["()"]
```

Yêu cầu: viết thêm một bản brute force bitmask cho hai loại ngoặc để **kiểm thử chéo** trên chuỗi ngẫu nhiên.

In [34]:
def remove_invalid_two_kinds(s: str) -> List[str]:
    # TODO 1: viết hàm kiểm tra hợp lệ cho hai loại ngoặc bằng stack
    # TODO 2: BFS theo tầng như mục 8, chỉ xóa các ký tự trong "()[]"
    pass

# Gợi ý test:
# assert sorted(remove_invalid_two_kinds("([)]")) == ["()", "[]"]
# assert remove_invalid_two_kinds("(a]") == ["a"]
# assert remove_invalid_two_kinds("()[]") == ["()[]"]
# assert remove_invalid_two_kinds("([)") == ["()"]

## 21. Gợi ý theo mức

**Bài 1**
- Mức 1: chạy tay đúng 3 quy tắc của mục 9: `'('` mở thêm, `')'` ghép nếu còn `'('` đang mở, ngược lại `right_rem += 1`.
- Mức 2: `"))(("`: hai `')'` đầu không ai ghép; hai `'('` cuối không ai đóng.
- Mức 3: độ dài đáp án = `len(s) - left_rem - right_rem`.

**Bài 2**
- Mức 1: tầng 1 sinh 4 chuỗi con (xóa từng index 0..3), nhưng xóa index 0 và index 1 cho cùng một chuỗi.
- Mức 2: tầng 1 là `["()(", "(((", "(()"]` — chưa chuỗi nào hợp lệ.
- Mức 3: sinh tầng 2 lần lượt từ `"()("`, `"((("`, `"(()"`; chỉ ghi chuỗi chưa có trong `visited`.

**Bài 3**
- Mức 1: lượt đếm chính là `count_removals`; phần DFS chính là `remove_invalid_dfs_set`.
- Mức 2: mỗi nhánh GIỮ có đủ 3 nhịp `path.append(c)` → `dfs(...)` → `path.pop()`.
- Mức 3: nhánh GIỮ `'('` gọi `dfs(i + 1, open_count + 1, del_left, del_right)`; nhánh GIỮ `')'` gọi `dfs(i + 1, open_count - 1, ...)` và nằm trong `if open_count > 0`.

**Bài 4**
- Mức 1: vòng `while True` với hai phần: "lọc hợp lệ" rồi "sinh tầng sau".
- Mức 2: `valid = [t for t in level if is_valid(t)]`, `if valid: return valid`.
- Mức 3: `child = t[:i] + t[i + 1:]`, chỉ khi `c in "()"` và `child not in visited`.

**Bài 5**
- Mức 1: các ngoặc phải xóa chính là những ngoặc mà lượt đếm của mục 9 coi là "không ai ghép".
- Mức 2: dùng một stack lưu **index** của các `'('` đang chờ; `')'` không ghép được thì đưa index vào tập `remove`.
- Mức 3: hết chuỗi, các index còn trong stack cũng vào `remove`; trả về `"".join(c for i, c in enumerate(s) if i not in remove)`.

**Bài 6**
- Mức 1: lượt đếm `left_rem/right_rem` không dùng được (`"([)]"`); BFS thì chỉ cần thay hàm kiểm tra.
- Mức 2: kiểm tra bằng stack: gặp ngoặc mở thì `push`; gặp ngoặc đóng thì đỉnh stack phải là ngoặc mở **cùng loại** (dùng dict `{")": "(", "]": "["}`), rồi `pop`; cuối cùng stack phải rỗng.
- Mức 3: brute force giống `remove_invalid_bruteforce`, chỉ đổi hàm kiểm tra; test ngẫu nhiên với bảng chữ `"()[]a"`.

## 22. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

**Bài 1:**

| Chuỗi | `(left_rem, right_rem)` | Độ dài mọi đáp án |
|---|:-:|:-:|
| `"())("` | `(1, 1)` | `4 - 2 = 2` (đáp án duy nhất `"()"`) |
| `"(a(b"` | `(2, 0)` | `4 - 2 = 2` (đáp án `"ab"`) |
| `"))(("` | `(2, 2)` | `0` (đáp án `""`) |
| `"a)b(c)d"` | `(0, 1)` | `7 - 1 = 6` (đáp án `"ab(c)d"`) |

**Bài 2:** `s = "(()("`

| Tầng | Sinh ra (theo thứ tự) | Sau khi lọc `visited` | Hợp lệ? |
|:-:|---|---|---|
| 0 | – | `"(()("` | ✘ |
| 1 | `"()("`, `"()("` (trùng), `"((("`, `"(()"` | `"()("`, `"((("`, `"(()"` | không chuỗi nào |
| 2 | từ `"()("`: `")("`, `"(("`, `"()"`; từ `"((("`: `"(("` ×3 (trùng); từ `"(()"`: `"()"`, `"()"`, `"(("` (đều trùng) | `")("`, `"(("`, `"()"` | `"()"` ✔ |

Dừng ở tầng 2 → xóa ít nhất 2 ký tự (khớp `count_removals("(()(") == (2, 0)`), đáp án `["()"]`. Cell dưới kiểm chứng cả hai bài.

In [35]:
# Kiểm chứng Bài 1
assert [count_removals(t) for t in ["())(", "(a(b", "))((", "a)b(c)d"]] == [(1, 1), (2, 0), (2, 2), (0, 1)]
assert Solution().removeInvalidParentheses("())(") == ["()"]
assert Solution().removeInvalidParentheses("(a(b") == ["ab"]
assert Solution().removeInvalidParentheses("a)b(c)d") == ["ab(c)d"]

# Kiểm chứng Bài 2: in lại các tầng bằng code
def bfs_levels(s: str) -> List[List[str]]:
    levels, level, visited = [], [s], {s}
    while True:
        levels.append(level)
        if any(is_valid(t) for t in level):
            return levels
        next_level = []
        for t in level:
            for i, c in enumerate(t):
                child = t[:i] + t[i + 1:]
                if c in "()" and child not in visited:
                    visited.add(child)
                    next_level.append(child)
        level = next_level

levels = bfs_levels("(()(")
for k, lv in enumerate(levels):
    print(f"Tầng {k}: {lv}")
assert levels == [["(()("], ["()(", "(((", "(()"], [")(", "((", "()"]]
assert remove_invalid_bfs("(()(") == ["()"]
print("Bài 1, Bài 2 PASS")

Tầng 0: ['(()(']
Tầng 1: ['()(', '(((', '(()']
Tầng 2: [')(', '((', '()']
Bài 1, Bài 2 PASS


In [36]:
# Bài 3
def answer_remove_invalid_dfs(s: str) -> List[str]:
    open_count = right_rem = 0
    for c in s:
        if c == "(":
            open_count += 1
        elif c == ")":
            if open_count > 0:
                open_count -= 1
            else:
                right_rem += 1
    left_rem = open_count
    n = len(s)
    result = set()
    path = []

    def dfs(i, open_count, del_left, del_right):
        if i == n:
            if del_left == 0 and del_right == 0 and open_count == 0:
                result.add("".join(path))
            return
        c = s[i]
        if c == "(":
            if del_left > 0:
                dfs(i + 1, open_count, del_left - 1, del_right)
            path.append(c)
            dfs(i + 1, open_count + 1, del_left, del_right)
            path.pop()
        elif c == ")":
            if del_right > 0:
                dfs(i + 1, open_count, del_left, del_right - 1)
            if open_count > 0:
                path.append(c)
                dfs(i + 1, open_count - 1, del_left, del_right)
                path.pop()
        else:
            path.append(c)
            dfs(i + 1, open_count, del_left, del_right)
            path.pop()

    dfs(0, 0, left_rem, right_rem)
    return list(result)


# Bài 4
def answer_bfs(s: str) -> List[str]:
    level = [s]
    visited = {s}
    while True:
        valid = [t for t in level if is_valid(t)]
        if valid:
            return valid
        next_level = []
        for t in level:
            for i, c in enumerate(t):
                if c in "()":
                    child = t[:i] + t[i + 1:]
                    if child not in visited:
                        visited.add(child)
                        next_level.append(child)
        level = next_level


assert sorted(answer_remove_invalid_dfs("()())()")) == ["(())()", "()()()"]
assert sorted(answer_remove_invalid_dfs("(a)())()")) == ["(a())()", "(a)()()"]
assert answer_remove_invalid_dfs(")(") == [""]
assert sorted(answer_bfs("()())()")) == ["(())()", "()()()"]
assert sorted(answer_bfs("(a)())()")) == ["(a())()", "(a)()()"]
assert answer_bfs(")(") == [""]
assert sorted(answer_bfs("(()(")) == ["()"]

random.seed(34)
for _ in range(500):
    s = "".join(random.choice("(()))x") for _ in range(random.randint(1, 11)))
    expected = sorted(remove_invalid_bruteforce(s))
    assert sorted(answer_remove_invalid_dfs(s)) == expected, s
    assert sorted(answer_bfs(s)) == expected, s
print("Bài 3, Bài 4 PASS (kể cả 500 test ngẫu nhiên so với brute force)")

Bài 3, Bài 4 PASS (kể cả 500 test ngẫu nhiên so với brute force)


In [37]:
# Bài 5 — LeetCode #1249
def answer_min_remove_to_make_valid(s: str) -> str:
    stack = []                 # index các '(' đang chờ ghép
    remove = set()             # index các ngoặc phải xóa
    for i, c in enumerate(s):
        if c == "(":
            stack.append(i)
        elif c == ")":
            if stack:
                stack.pop()            # ghép với '(' gần nhất
            else:
                remove.add(i)          # ')' không có ai ghép
    remove.update(stack)               # '(' không bao giờ được đóng
    return "".join(c for i, c in enumerate(s) if i not in remove)


for s in ["lee(t(c)o)de)", "a)b(c)d", "))((", "(a(b(c)d)"]:
    ans = answer_min_remove_to_make_valid(s)
    print(f"{s!r:16} -> {ans!r}")
    assert is_valid(ans) and len(ans) == len(s) - sum(count_removals(s))
    assert ans in set(Solution().removeInvalidParentheses(s))


# Bài 6 — hai loại ngoặc
def is_valid_two_kinds(t: str) -> bool:
    match = {")": "(", "]": "["}
    stack = []
    for c in t:
        if c in "([":
            stack.append(c)
        elif c in ")]":
            if not stack or stack[-1] != match[c]:
                return False
            stack.pop()
    return not stack


def answer_remove_invalid_two_kinds(s: str) -> List[str]:
    level, visited = [s], {s}
    while True:
        valid = [t for t in level if is_valid_two_kinds(t)]
        if valid:
            return valid
        next_level = []
        for t in level:
            for i, c in enumerate(t):
                if c in "()[]":
                    child = t[:i] + t[i + 1:]
                    if child not in visited:
                        visited.add(child)
                        next_level.append(child)
        level = next_level


def brute_two_kinds(s: str) -> List[str]:
    n, best_len, best = len(s), -1, set()
    for mask in range(1 << n):
        t = "".join(s[i] for i in range(n) if (mask >> i) & 1)
        if is_valid_two_kinds(t):
            if len(t) > best_len:
                best_len, best = len(t), {t}
            elif len(t) == best_len:
                best.add(t)
    return list(best)


assert sorted(answer_remove_invalid_two_kinds("([)]")) == ["()", "[]"]
assert answer_remove_invalid_two_kinds("(a]") == ["a"]
assert answer_remove_invalid_two_kinds("()[]") == ["()[]"]
assert answer_remove_invalid_two_kinds("([)") == ["()"]
random.seed(36)
for _ in range(800):
    s = "".join(random.choice("()[]a") for _ in range(random.randint(1, 10)))
    assert sorted(answer_remove_invalid_two_kinds(s)) == sorted(brute_two_kinds(s)), s
print("Bài 5, Bài 6 PASS (kể cả 800 test ngẫu nhiên cho Bài 6)")

'lee(t(c)o)de)'  -> 'lee(t(c)o)de'
'a)b(c)d'        -> 'ab(c)d'
'))(('           -> ''
'(a(b(c)d)'      -> 'a(b(c)d)'


Bài 5, Bài 6 PASS (kể cả 800 test ngẫu nhiên cho Bài 6)


## 23. Template ngắn nên nhớ

**Câu thần chú cho cách chính:** *"Đếm thừa — giữ/xóa từng ký tự — đủ hạn mức thì thôi xóa — hết mở thì cấm giữ `')'` — trùng đầu dãy thì cấm xóa."*

```python
class Solution:
    def removeInvalidParentheses(self, s: str) -> List[str]:
        opened = right_rem = 0                       # 1) đếm ngoặc thừa
        for c in s:
            if c == "(":
                opened += 1
            elif c == ")":
                if opened: opened -= 1
                else: right_rem += 1
        left_rem = opened

        n, result, path = len(s), [], []
        def dfs(i, opened, dl, dr, prev_kept):      # 2) DFS giữ/xóa
            if i == n:
                if dl == dr == opened == 0:
                    result.append("".join(path))
                return
            c = s[i]
            can_del = not (i and s[i - 1] == c and prev_kept)
            if c == "(" and dl and can_del:
                dfs(i + 1, opened, dl - 1, dr, False)
            if c == ")" and dr and can_del:
                dfs(i + 1, opened, dl, dr - 1, False)
            if c != ")" or opened:                  # giữ (P2 cho ')')
                path.append(c)
                dfs(i + 1, opened + (c == "(") - (c == ")"), dl, dr, True)
                path.pop()
        dfs(0, 0, left_rem, right_rem, False)
        return result
```

**Câu thần chú cho cách 4:** *"Âm thì xóa một `')'` đầu dãy từ `last_j` tới `i`, đệ quy từ `(i, j)` rồi `return`; quét sạch thì đảo chuỗi, đổi vai; sạch cả hai chiều thì đảo lại, ghi nhận."*

**BFS dự phòng (tổng quát nhất):** *"Tầng `k` = xóa `k` ngoặc; tầng đầu tiên có chuỗi hợp lệ thì trả trọn tầng."*

Cell cuối cùng kiểm tra bản template gọn ở trên so với brute force, để chắc chắn bản rút gọn không lệch logic.

In [38]:
class TemplateSolution:
    def removeInvalidParentheses(self, s: str) -> List[str]:
        opened = right_rem = 0
        for c in s:
            if c == "(":
                opened += 1
            elif c == ")":
                if opened: opened -= 1
                else: right_rem += 1
        left_rem = opened

        n, result, path = len(s), [], []
        def dfs(i, opened, dl, dr, prev_kept):
            if i == n:
                if dl == dr == opened == 0:
                    result.append("".join(path))
                return
            c = s[i]
            can_del = not (i and s[i - 1] == c and prev_kept)
            if c == "(" and dl and can_del:
                dfs(i + 1, opened, dl - 1, dr, False)
            if c == ")" and dr and can_del:
                dfs(i + 1, opened, dl, dr - 1, False)
            if c != ")" or opened:
                path.append(c)
                dfs(i + 1, opened + (c == "(") - (c == ")"), dl, dr, True)
                path.pop()
        dfs(0, 0, left_rem, right_rem, False)
        return result


random.seed(2301)
for s, expected in tests:
    assert sorted(TemplateSolution().removeInvalidParentheses(s)) == sorted(expected)
for _ in range(1000):
    s = "".join(random.choice("(()))ab") for _ in range(random.randint(1, 12)))
    got = TemplateSolution().removeInvalidParentheses(s)
    assert sorted(got) == sorted(remove_invalid_bruteforce(s)) and len(got) == len(set(got)), s
print("Template PASS: khớp test cố định và 1000 test ngẫu nhiên")

Template PASS: khớp test cố định và 1000 test ngẫu nhiên


## 24. Checklist trước khi submit

- [ ] Hàm kiểm tra hợp lệ **bỏ qua chữ cái** (dùng `elif c == ")"`, không dùng `else`)?
- [ ] Trả về **mọi** chuỗi với số lần xóa **ít nhất** — không thiếu (BFS trả về sớm), không thừa (BFS không dừng)?
- [ ] Kết quả **không trùng** (dùng `set`, hoặc quy tắc "chỉ xóa đầu dãy" của cách 3 / cách 4)?
- [ ] `")("` trả về `[""]` (danh sách chứa chuỗi rỗng), **không phải** `[]`?
- [ ] Cách 3: đã đếm `left_rem`, `right_rem` trước; nhánh xóa có kiểm hạn mức; nhánh giữ `')'` chỉ khi `open_count > 0`; luôn `path.pop()` sau khi đi sâu?
- [ ] Cách 4: có `return` sau vòng `for j`; điều kiện `j == last_j or t[j - 1] != pair[1]`; lượt ngược ghi nhận `t[::-1]`?
- [ ] BFS: chỉ xóa **ngoặc**, có `visited`, kiểm tra **trọn tầng** rồi mới dừng?
- [ ] Đã thử: 3 ví dụ của đề, `"abc"`, `"(((("`, `"))))"`, `"x("`, `"(((k()(("`, và so với brute force trên chuỗi ngẫu nhiên ngắn?
- [ ] Độ phức tạp: hàm mũ theo số ngoặc `p ≤ 20`, nhưng nhờ đếm trước + cắt tỉa thì chạy rất nhanh trong giới hạn đề.